# Case Study Group 44 - Factory Audit Recommendation

**Course:** Introduction to Engineering Data Analytics  
**Semester:** Summer Semester 2026  
**Group number:** 44  
**Case study scope:** OEM1 Vehicle Type 11 and OEM2 Vehicle Type 21

# 1. Project Information

## 1.1 Group Members

| Name | Matr.-Nr. |
|---|---|
| Group member 1 | `[TO BE ADDED]` |
| Group member 2 | `[TO BE ADDED]` |
| Group member 3 | `[TO BE ADDED]` |
| Group member 4 | `[TO BE ADDED]` |
| Group member 5 | `[TO BE ADDED]` |

## 1.2 Purpose of This Notebook

The notebook serves as both the analytical workflow and the written case study report. It contains all code required to reproduce the analysis from the original source files, together with explanations of the analytical decisions, validation steps, visualisations, and interpretations of the results.

## 1.3 Project Deliverables

The final case study submission consists of the following files:

- `SoSe26_Case_Study_Group_44.ipynb`: executable analysis and report,
- `SoSe26_Case_Study_Group_44.html`: static export with all final outputs,
- `SoSe26_Case_Study_finalData_Group_44.csv`: final processed dataset used by the application, and
- `SoSe26_Case_Study_App_Group_44.py`: interactive web application.

All project paths are defined relative to the submission folder so that the complete workflow can be run on another computer without modifying computer-specific paths.

# 2. Business Context, Task, and Analytical Strategy

## 2.1 Business Context

The automotive group manufactures several vehicle types under the brands OEM1 and OEM2. Its supply chain comprises Tier-2 suppliers that manufacture individual parts, Tier-1 suppliers that assemble these parts into components, and OEM production plants that install the components in finished vehicles. A quality issue recorded at any of these levels can therefore affect the quality status of a completed vehicle.

We must decide where the next process audit should take place. The decision should reflect both the number of affected vehicles in the field and the defect frequency relative to the production volume of each plant.

## 2.2 Assigned Task

> Analyse the failure data for OEM1 Vehicle Type 11 and OEM2 Vehicle Type 21. Relate the absolute and relative defect frequencies of these vehicles to their production plants and recommend the plant in which the next process audit should be conducted. Identify additional insights that can support the audit.

## 2.3 Analytical Objective

The primary objective is to identify the OEM production plant with the greatest audit priority. The main decision criterion is the relative frequency of defective vehicles because it relates observed defects to the number of vehicles produced at each plant. Absolute defect counts are analysed alongside the relative rates mainly to represent customer impact.

The analysis also examines defect sources at component and individual-part level. These supplier-level results are used to refine the proposed audit scope, but they do not replace the OEM plant comparison required by the task.

## 2.4 Defect Definition

Three related defect indicators are distinguished:

- **Vehicle defect:** the defect flag recorded directly in the vehicle production table.
- **Component defect:** a defect flag recorded for at least one component installed in the vehicle.
- **Individual-part defect:** a defect flag recorded for at least one part contained in an installed component.

A vehicle is classified as defective when at least one of these indicators confirms a defect. This definition sets the rule that defects propagate through the supply chain from individual parts to components and from components to the finished vehicle. Missing defect information remains explicitly unknown during data preparation and is examined before the final rates are interpreted.

## 2.5 Analytical Questions

The analysis is organised around five questions:

1. How many defective vehicles are associated with each OEM production plant?
2. What proportion of the vehicles produced at each plant is effectively defective?
3. Does the plant ranking change when only direct vehicle defects are considered instead of the complete supply-chain definition?
4. Which component roles, component types, supplier plants, and individual parts contribute the most to vehicle defects?
5. How do the observed defect patterns change over time, and which limitations must be considered when translating them into an audit recommendation?

## 2.6 Analytical Strategy

The workflow follows the logic of the business question. First, the relevant vehicle, component, part, mapping, and plant tables are identified. Second, the raw data are imported and examined before major transformations. Third, recurring quality problems are cleaned systematically and the supply-chain tables are integrated using validated keys. Then, one  final dataset is created on a unique vehicle level using the defined defect-propagation rule. Finally, the OEM plants are compared using absolute counts and relative rates.

# 3. Data Scope and Data Model

## 3.1 Scope of the Analysis

The required analysis consists of **OEM1 Vehicle Type 11** and **OEM2 Vehicle Type 21**. Other vehicle types, registrations, and logistics-delay tables are outside the scope because they are not required to answer the assigned plant-audit question. Geodata are included only where they identify the production locations of vehicles, components, and individual parts.

The unit of analysis for the final OEM comparison is one finished vehicle. Supplier tables contain lower-level observations and are integrated only to determine whether an installed component or individual part changes the effective defect status of that vehicle.

## 3.2 Data-Selection Strategy

Data selection starts with the two required vehicle master files and their vehicle-component mapping files. The component identifiers in these mappings determine the relevant body, transmission, seat, and engine variants. The component-part mappings then determine which individual-part types belong to those components. Finally, the OEM, Tier-1, and Tier-2 plant tables provide a common location reference for the three supply-chain levels.

The resulting inventory is reported below so that the import is reproducible.

## 3.3 Supply-Chain Data Model

The data has the following structure:

**Individual part -> component -> vehicle -> OEM production plant**

| Data level | Single unit (Row) | Primary key (Column) | Purpose |
|---|---|---|---|
| Vehicle | One finished Type 11 or Type 21 vehicle | `ID_Fahrzeug` | Defines the final unit of analysis, OEM plant, production date, and direct vehicle-defect status |
| Vehicle-component mapping | One vehicle and its installed components | `ID_Fahrzeug` and component ID columns | Connects each vehicle to body, transmission, seat, and engine components |
| Component | One component | `ID_Komponente` | Provides component type, Tier-1 plant, production information, and component-defect status |
| Component-part mapping | One component and its installed parts | `ID_Komponente` and part ID columns | Connects components to their individual parts |
| Individual part | One individual part | `ID_Einzelteil` | Provides part type, Tier-2 plant, production information, and part-defect status |
| Plant | One OEM, Tier-1, or Tier-2 production location | `Werksnummer` | Adds the plant name, city, postal code, and geographical coordinates |

## 3.4 Relationships, Merge Keys, and Validation Expectations

| Relationship | Merge key | Expected relationship after reshaping | Validation objective |
|---|---|---|---|
| Vehicle data -> vehicle-component assignments | `ID_Fahrzeug` | One-to-many | Every vehicle should be matched to four components |
| Vehicle-component assignments -> component data | `ID_Komponente` | Many-to-one | Every installed component should have a unique component record |
| Component data -> component-part assignments | `ID_Komponente` | One-to-many | Every component should be connected to its part types |
| Component-part assignments -> part data | `ID_Einzelteil` | Many-to-one | Every installed part should have a unique part record |
| Vehicles, components, and parts -> plant data | `Werksnummer` | Many-to-one | Every production object should be assigned to the correct plant in the supply chain |

The integration is successful when merged tables match these expectations, identifiers remain complete, and row counts can be reconciled before and after each join.

## 3.5 Assumptions and Exclusions

- The structured object identifiers are used to verify manufacturer and plant numbers when separately stored values are missing or inconsistent.
- Only component and part variants that occur in the mappings of Vehicle Types 11 and 21 are relevant to the analysis.
- Registration and logistics-delay data are excluded because the assigned task asks for production-plant comparisons based on vehicle defects, not registration or logistics performance.
- Supplier-level defect rates describe where components and parts were produced. They offer additional insights but are not within the scope of the auditing decision.

## 3.6 Source-File Inventory

These are the source files required to reconstruct the complete vehicle-component-part quality chain for both vehicle types.

### 3.6.1 Relevant Files for Vehicle Type 11

| Category | Filename |
|---|---|
| Vehicle data | `Fahrzeuge_OEM1_Typ11.csv` |
| Vehicle–component mapping | `Bestandteile_Fahrzeuge_OEM1_Typ11.csv` |
| OEM plant locations | `OEM_Werke_2017-07-04_TrR.csv` |
| Tier 1 plant locations | `Tier1_Werke_2017-07-11_v1.2_TrR.csv` |
| Tier 2 plant locations | `Tier2_Werke_2017-07-11_v1.2_TrR.csv` |
| Body component | `Komponente_K4.csv` |
| Body component–part mapping | `Bestandteile_Komponente_K4.csv` |
| Body part | `Einzelteil_T30.csv` |
| Body part | `Einzelteil_T31.txt` |
| Body part | `Einzelteil_T32.csv` |
| Transmission component | `Komponente_K3AG1.csv` |
| Transmission component | `Komponente_K3SG1.csv` |
| Transmission component–part mapping | `Bestandteile_Komponente_K3AG1.csv` |
| Transmission component–part mapping | `Bestandteile_Komponente_K3SG1.csv` |
| Transmission part | `Einzelteil_T21.csv` |
| Transmission part | `Einzelteil_T22.txt` |
| Transmission part | `Einzelteil_T23.csv` |
| Transmission part | `Einzelteil_T24.txt` |
| Transmission part | `Einzelteil_T25.csv` |
| Seat component | `Komponente_K2LE1.txt` |
| Seat component | `Komponente_K2ST1.txt` |
| Seat component–part mapping | `Bestandteile_Komponente_K2LE1.csv` |
| Seat component–part mapping | `Bestandteile_Komponente_K2ST1.csv` |
| Seat part | `Einzelteil_T11.txt` |
| Seat part | `Einzelteil_T12.csv` |
| Seat part | `Einzelteil_T13.csv` |
| Seat part | `Einzelteil_T14.csv` |
| Seat part | `Einzelteil_T15.csv` |
| Engine component | `Komponente_K1BE1.csv` |
| Engine component | `Komponente_K1DI1.csv` |
| Engine component–part mapping | `Bestandteile_Komponente_K1BE1.csv` |
| Engine component–part mapping | `Bestandteile_Komponente_K1DI1.csv` |
| Engine part | `Einzelteil_T01.txt` |
| Engine part | `Einzelteil_T02.txt` |
| Engine part | `Einzelteil_T03.txt` |
| Engine part | `Einzelteil_T04.csv` |
| Engine part | `Einzelteil_T05.csv` |
| Engine part | `Einzelteil_T06.csv` |

### 3.6.2 Relevant Files for Vehicle Type 21

| Category | Filename |
|---|---|
| Vehicle data | `Fahrzeuge_OEM2_Typ21.csv` |
| Vehicle–component mapping | `Bestandteile_Fahrzeuge_OEM2_Typ21.csv` |
| OEM plant locations | `OEM_Werke_2017-07-04_TrR.csv` |
| Tier 1 plant locations | `Tier1_Werke_2017-07-11_v1.2_TrR.csv` |
| Tier 2 plant locations | `Tier2_Werke_2017-07-11_v1.2_TrR.csv` |
| Body component | `Komponente_K6.csv` |
| Body component–part mapping | `Bestandteile_Komponente_K6.csv` |
| Body part | `Einzelteil_T34.txt` |
| Body part | `Einzelteil_T35.txt` |
| Body part | `Einzelteil_T36.txt` |
| Body part | `Einzelteil_T37.csv` |
| Transmission component | `Komponente_K3AG2.txt` |
| Transmission component | `Komponente_K3SG2.csv` |
| Transmission component–part mapping | `Bestandteile_Komponente_K3AG2.csv` |
| Transmission component–part mapping | `Bestandteile_Komponente_K3SG2.csv` |
| Transmission part | `Einzelteil_T21.csv` |
| Transmission part | `Einzelteil_T22.txt` |
| Transmission part | `Einzelteil_T24.txt` |
| Transmission part | `Einzelteil_T26.csv` |
| Transmission part | `Einzelteil_T27.txt` |
| Seat component | `Komponente_K2LE2.txt` |
| Seat component | `Komponente_K2ST2.csv` |
| Seat component–part mapping | `Bestandteile_Komponente_K2LE2.csv` |
| Seat component–part mapping | `Bestandteile_Komponente_K2ST2.csv` |
| Seat part | `Einzelteil_T16.txt` |
| Seat part | `Einzelteil_T17.csv` |
| Seat part | `Einzelteil_T18.csv` |
| Seat part | `Einzelteil_T19.csv` |
| Seat part | `Einzelteil_T20.txt` |
| Engine component | `Komponente_K1BE2.csv` |
| Engine component | `Komponente_K1DI2.txt` |
| Engine component–part mapping | `Bestandteile_Komponente_K1BE2.csv` |
| Engine component–part mapping | `Bestandteile_Komponente_K1DI2.csv` |
| Engine part | `Einzelteil_T01.txt` |
| Engine part | `Einzelteil_T02.txt` |
| Engine part | `Einzelteil_T07.txt` |
| Engine part | `Einzelteil_T08.csv` |
| Engine part | `Einzelteil_T09.txt` |
| Engine part | `Einzelteil_T10.csv` |

# 4. Data Import and Initial Exploration

The files required by the data scope are listed in Section 3. Before any cleaning, we:

1. define all paths relative to the project folder,
2. check that every required file is actually present,
3. open a few representative raw files to see typical formatting problems.

The goal is to understand the raw data as it currently exists, so the later required cleaning steps can be found out.


## 4.1 Imports

`pathlib` keeps file paths independent of the operating system. `pandas` is used for all tables. `csv`, `re`, and `mmap` are needed later for a few TXT files that do not use a normal delimiter or line break. `numpy` is used for everything numeric.


In [ ]:
from pathlib import Path
import csv
import mmap
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.max_rows", 100)


## 4.2 Project and Data Directories

All paths start from the current project folder.

The helper function `check_paths` is reused for every file group below. It shows whether a listed path exists without opening the file.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
DATA_DIRECTORY = PROJECT_ROOT / "data"

VEHICLE_DIRECTORY = DATA_DIRECTORY / "Fahrzeug"
COMPONENT_DIRECTORY = DATA_DIRECTORY / "Komponente"
PART_DIRECTORY = DATA_DIRECTORY / "Einzelteil"
GEODATA_DIRECTORY = DATA_DIRECTORY / "Geodaten"


def check_paths(path_dict, expect_file=True):
    """Show whether each listed path exists."""
    rows = []
    for name, path in path_dict.items():
        exists = path.is_file() if expect_file else path.is_dir()
        rows.append(
            {
                "File": name,
                "Relative Path": path.relative_to(PROJECT_ROOT).as_posix(),
                "Exists": "Yes" if exists else "No",
            }
        )
    return pd.DataFrame(rows)


data_directories = {
    "Vehicle data": VEHICLE_DIRECTORY,
    "Component data": COMPONENT_DIRECTORY,
    "Part data": PART_DIRECTORY,
    "Plant data": GEODATA_DIRECTORY,
}

check_paths(data_directories, expect_file=False)


All four data folders are available. The same overview is used for every file group below.

## 4.3 Vehicle Files

Each vehicle type has one vehicle file and one mapping file that links the vehicle to its four installed components.


In [ ]:
vehicle_paths = {
    "Vehicle Type 11": VEHICLE_DIRECTORY / "Fahrzeuge_OEM1_Typ11.csv",
    "Vehicle Type 11 mapping": VEHICLE_DIRECTORY / "Bestandteile_Fahrzeuge_OEM1_Typ11.csv",
    "Vehicle Type 21": VEHICLE_DIRECTORY / "Fahrzeuge_OEM2_Typ21.csv",
    "Vehicle Type 21 mapping": VEHICLE_DIRECTORY / "Bestandteile_Fahrzeuge_OEM2_Typ21.csv",
}

check_paths(vehicle_paths)


## 4.4 Component Files

For each component we need two files: the production file (quality and plant of the component) and the mapping file (which parts are installed in it).


In [ ]:
component_paths = {
    "K4 production": COMPONENT_DIRECTORY / "Komponente_K4.csv",
    "K4 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K4.csv",
    "K3AG1 production": COMPONENT_DIRECTORY / "Komponente_K3AG1.csv",
    "K3AG1 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K3AG1.csv",
    "K3SG1 production": COMPONENT_DIRECTORY / "Komponente_K3SG1.csv",
    "K3SG1 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K3SG1.csv",
    "K2LE1 production": COMPONENT_DIRECTORY / "Komponente_K2LE1.txt",
    "K2LE1 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K2LE1.csv",
    "K2ST1 production": COMPONENT_DIRECTORY / "Komponente_K2ST1.txt",
    "K2ST1 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K2ST1.csv",
    "K1BE1 production": COMPONENT_DIRECTORY / "Komponente_K1BE1.csv",
    "K1BE1 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K1BE1.csv",
    "K1DI1 production": COMPONENT_DIRECTORY / "Komponente_K1DI1.csv",
    "K1DI1 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K1DI1.csv",
    "K6 production": COMPONENT_DIRECTORY / "Komponente_K6.csv",
    "K6 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K6.csv",
    "K3AG2 production": COMPONENT_DIRECTORY / "Komponente_K3AG2.txt",
    "K3AG2 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K3AG2.csv",
    "K3SG2 production": COMPONENT_DIRECTORY / "Komponente_K3SG2.csv",
    "K3SG2 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K3SG2.csv",
    "K2LE2 production": COMPONENT_DIRECTORY / "Komponente_K2LE2.txt",
    "K2LE2 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K2LE2.csv",
    "K2ST2 production": COMPONENT_DIRECTORY / "Komponente_K2ST2.csv",
    "K2ST2 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K2ST2.csv",
    "K1BE2 production": COMPONENT_DIRECTORY / "Komponente_K1BE2.csv",
    "K1BE2 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K1BE2.csv",
    "K1DI2 production": COMPONENT_DIRECTORY / "Komponente_K1DI2.txt",
    "K1DI2 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K1DI2.csv",
}

check_paths(component_paths)


## 4.5 Part Files

Each part type is listed only once, even if both vehicle types use it. That avoids reading the same file twice later.


In [ ]:
part_paths = {
    "T01": PART_DIRECTORY / "Einzelteil_T01.txt",
    "T02": PART_DIRECTORY / "Einzelteil_T02.txt",
    "T03": PART_DIRECTORY / "Einzelteil_T03.txt",
    "T04": PART_DIRECTORY / "Einzelteil_T04.csv",
    "T05": PART_DIRECTORY / "Einzelteil_T05.csv",
    "T06": PART_DIRECTORY / "Einzelteil_T06.csv",
    "T07": PART_DIRECTORY / "Einzelteil_T07.txt",
    "T08": PART_DIRECTORY / "Einzelteil_T08.csv",
    "T09": PART_DIRECTORY / "Einzelteil_T09.txt",
    "T10": PART_DIRECTORY / "Einzelteil_T10.csv",
    "T11": PART_DIRECTORY / "Einzelteil_T11.txt",
    "T12": PART_DIRECTORY / "Einzelteil_T12.csv",
    "T13": PART_DIRECTORY / "Einzelteil_T13.csv",
    "T14": PART_DIRECTORY / "Einzelteil_T14.csv",
    "T15": PART_DIRECTORY / "Einzelteil_T15.csv",
    "T16": PART_DIRECTORY / "Einzelteil_T16.txt",
    "T17": PART_DIRECTORY / "Einzelteil_T17.csv",
    "T18": PART_DIRECTORY / "Einzelteil_T18.csv",
    "T19": PART_DIRECTORY / "Einzelteil_T19.csv",
    "T20": PART_DIRECTORY / "Einzelteil_T20.txt",
    "T21": PART_DIRECTORY / "Einzelteil_T21.csv",
    "T22": PART_DIRECTORY / "Einzelteil_T22.txt",
    "T23": PART_DIRECTORY / "Einzelteil_T23.csv",
    "T24": PART_DIRECTORY / "Einzelteil_T24.txt",
    "T25": PART_DIRECTORY / "Einzelteil_T25.csv",
    "T26": PART_DIRECTORY / "Einzelteil_T26.csv",
    "T27": PART_DIRECTORY / "Einzelteil_T27.txt",
    "T30": PART_DIRECTORY / "Einzelteil_T30.csv",
    "T31": PART_DIRECTORY / "Einzelteil_T31.txt",
    "T32": PART_DIRECTORY / "Einzelteil_T32.csv",
    "T34": PART_DIRECTORY / "Einzelteil_T34.txt",
    "T35": PART_DIRECTORY / "Einzelteil_T35.txt",
    "T36": PART_DIRECTORY / "Einzelteil_T36.txt",
    "T37": PART_DIRECTORY / "Einzelteil_T37.csv",
}

check_paths(part_paths)


## 4.6 Plant Files

The three plant files connect manufacturer and plant numbers to a location. OEM plants assemble vehicles, Tier 1 plants produce components, and Tier 2 plants produce parts.


In [ ]:
plant_paths = {
    "OEM plants": GEODATA_DIRECTORY / "OEM_Werke_2017-07-04_TrR.csv",
    "Tier 1 plants": GEODATA_DIRECTORY / "Tier1_Werke_2017-07-11_v1.2_TrR.csv",
    "Tier 2 plants": GEODATA_DIRECTORY / "Tier2_Werke_2017-07-11_v1.2_TrR.csv",
}

check_paths(plant_paths)


## 4.7 Final Path Check

If any listed file is missing, the notebook stops here.


In [ ]:
# Combine every listed source so a missing file stops the notebook immediately.
all_relevant_paths = {
    **vehicle_paths,
    **component_paths,
    **part_paths,
    **plant_paths,
}

missing_files = [
    path.relative_to(PROJECT_ROOT).as_posix()
    for path in all_relevant_paths.values()
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(f"Missing files: {missing_files}")

print(f"All {len(all_relevant_paths)} required files are available.")


## 4.8 Selected Raw File Previews

### 4.8.1 Vehicle Type 11

Five rows of the Type 11 vehicle file, read with its comma separator.


In [ ]:
vehicle_type_11_raw = pd.read_csv(
    vehicle_paths["Vehicle Type 11"],
    sep=",",
    encoding="cp1252",
    nrows=5,
)
vehicle_type_11_raw.head()


The first two columns are an unnamed column and `X1`. They look like duplicate row counters and will need to be removed.

### 4.8.2 Component K4

Component K4 is an example of a file with duplicated column groups.


In [ ]:
component_k4_raw = pd.read_csv(
    component_paths["K4 production"],
    sep=";",
    encoding="cp1252",
    nrows=5,
)
component_k4_raw.head()


Many attributes appear once with the suffix `.x` and again with `.y`. In the displayed rows the `.y` columns appear to be empty. During cleaning, both versions have to be compared first before they are merged into a single column. Again there are an unnamed column and `X1` that need to be removed.

### 4.8.3 Special TXT format

Several component and part files are not regular CSVs. `K2LE1` is one example.


In [ ]:
def show_raw_excerpt(file_path, byte_limit=500):
    """Read a short raw excerpt and make control characters visible."""
    with file_path.open("rb") as raw_file:
        raw_text = raw_file.read(byte_limit).decode("cp1252", errors="replace")

    # Replace invisible control characters with readable labels.
    raw_text = raw_text.replace("\x08", "<BACKSPACE>")
    raw_text = raw_text.replace("\x0b", "<VERTICAL_TAB>")
    raw_text = raw_text.replace("\r", "<CR>")
    raw_text = raw_text.replace("\n", "<LF>")
    return raw_text


show_raw_excerpt(component_paths["K2LE1 production"])


Fields are separated by `II`, while records (rows) are separated by `<VERTICAL_TAB>`. A normal `read_csv` would not produce the correct columns and rows. These files need the dedicated import step in Section 5.2.


## 4.9 Summary of Observed Data Issues

### 4.9.1 File Import Problems

| Problem | Affected Files | Required Cleaning Step |
|---|---|---|
| CSV files use different separators | **Comma-separated:** `Fahrzeuge_OEM1_Typ11.csv`, `Fahrzeuge_OEM2_Typ21.csv`, `Komponente_K1BE1.csv`, `Komponente_K1DI1.csv`, `Komponente_K3AG1.csv`, `Komponente_K3SG1.csv`, `Komponente_K3SG2.csv`, `Einzelteil_T05.csv`, `T06.csv`, `T08.csv`, `T19.csv`, `T25.csv`, `T30.csv`, `T37.csv`<br><br>**Semicolon-separated:** all other relevant CSV files | Load each file with the correct separator. Then check whether the columns were imported correctly. |
| Component TXT files use special separators | `Komponente_K2LE1.txt`: fields `II`, rows `\x0b`<br>`Komponente_K3AG2.txt`: fields `\`, rows `\r`<br>`Komponente_K2LE2.txt`: fields `\`, rows `\r`<br>`Komponente_K2ST1.txt`: fields `\|`, rows `\r`<br>`Komponente_K1DI2.txt`: fields `\`, rows `\t` | Store the field and row separator for each file and use the special TXT reader to import it. |
| Part TXT files use special row separators | `T02`: fields = 2+ spaces, rows = `\t`<br>`T03`: fields = `\|`, rows = `\x0b`<br>`T09`: fields = `\`, rows = `\x0b`<br>`T11`: fields = `\t`, rows = `\x0c`<br>`T16`: fields = ` \| \| `, rows = `\t`<br>`T24`: fields = 2+ spaces, rows = `\x0c`<br>`T27`: fields = ` \| \| `, rows = `\x07`<br>`T31`: fields = 2+ spaces, rows = `\x08` | First split the file with the row separator. Then split each row with the field separator. |
| Some part TXT files have no row separator | `T01`: fields = ` \| \| `<br>`T07`: fields = `\t`<br>`T20`: fields = ` \| \| `<br>`T22`: fields = `\t`<br>`T34`: fields = ` \| \| `<br>`T35`: fields = `\`<br>`T36`: fields = 2+ spaces | Find the start of each new row using the quoted row number and split the file at these positions. |
| TXT rows can have the wrong number of fields | All component and part TXT files with special separators | Compare the number of values in every row with the number of header columns. Stop the import if they do not match. |
| Source files use a different text encoding | All imported source files | Read the files with `cp1252` encoding and replace characters that cannot be decoded. |

### 4.9.2 Structural and Content Cleaning

| Problem | Affected Files | Required Cleaning Step |
|---|---|---|
| Unnecessary export columns | Vehicle, component, assignment, and part files; columns such as `X`, `X1`, `_Zeilenindex`, empty column names, or `Unnamed:*` | Remove columns with these names after importing the file. |
| Extra spaces and different missing-value markers | All imported files | Remove spaces around column names and text values. Replace `""`, `NA`, `N/A`, `NULL`, and `null` with `pd.NA`. |
| The same data is stored in `.x` and `.y` columns | **Components:** `K4`, `K3AG1`, `K3SG1`, `K2LE1`, `K1DI1`<br><br>**Parts:** `T01`, `T02`, `T05`, `T09`, `T12`, `T15`, `T16`, `T17`, `T22`, `T23`, `T24`, `T30`, `T32`, `T35` | Combine the normal, `.x`, and `.y` versions into one column. |
| Production dates are stored differently | `Fahrzeuge_OEM2_Typ21.csv`<br><br>Components `K1BE1`, `K6`, `K3AG2`, `K3SG2`, `K2LE2`, `K2ST2`, `K1BE2`, `K1DI2`<br><br>Parts `T03`, `T04`, `T06`, `T07`, `T08`, `T10`, `T11`, `T13`, `T14`, `T18`, `T19`, `T20`, `T21`, `T25`, `T26`, `T27`, `T31`, `T34`, `T36`, `T37` | Convert normal date values directly. For origin-based dates, add the number of days to the date in `origin`. Save the result as `YYYY-MM-DD`. |
| Defect dates use different formats | Vehicle, component, and part files containing `Fehlerhaft_Datum` | Convert the values to dates and save them as `YYYY-MM-DD`. |
| IDs and keys may be read as numbers | All ID, manufacturer, plant, and postal-code columns | Read the columns as text. |
| Manufacturer or plant numbers do not match the object ID | Vehicle, component, and part files with IDs in the format `Type-Manufacturer-Plant-…` | Get the manufacturer and plant number from the object ID. Use these values when the separate columns are missing or different. |
| Plant files contain empty or inconsistent values | OEM, Tier-1, and Tier-2 plant files | Remove completely empty columns and rows without plant, city, or postal code. Fix the longitude column name, convert coordinates to numbers, and fill postal codes to five digits. |
| OEM plant numbers have a different format | OEM plant data | Remove the `O` at the start of values such as `O11`. This creates the same plant-number format as in the vehicle data. |
| Vehicle-component assignments are stored in several columns | `Bestandteile_Fahrzeuge_OEM1_Typ11.csv`, `Bestandteile_Fahrzeuge_OEM2_Typ21.csv` | Change the four component columns into a long table with one vehicle-component pair per row. Add the component role and component type. |
| Part ID columns use different names | Component-part assignment files; for example `ID_T1` and `ID_T01` | Check both possible column names and rename the selected column to `ID_Einzelteil`. |
| Different components require different part types | All component-part assignment files | Use the configured list of part types for each component type. Only these columns are added to the component-part table. |
| Some component IDs occur more than once | Component master files | Group rows with the same component ID. Keep the highest defect value and the first value from the other columns. Record the number of duplicate rows. |
| Some part IDs occur more than once | Part master files | Group rows with the same part ID. Keep the highest defect value and the first value from the other columns. Record the number of duplicate rows. |
| Defect and mileage columns have the wrong data type | Vehicle, component, and part files | Convert `Fehlerhaft` to a nullable integer and `Fehlerhaft_Fahrleistung` to a number. |
| Plant information is stored in separate files | Vehicle, component, and part data | Add OEM plant data to vehicles, Tier-1 plant data to components, and Tier-2 plant data to parts using `Werksnummer`. |

### 4.9.3 Data Quality and Validation

| Problem | Affected Files | Required Cleaning or Validation Step |
|---|---|---|
| Missing defect values must stay unknown | Vehicle, component, and part data | Use three defect states: `1` for defective, `0` for not defective, and `NA` for unknown. |
| Component and part defects also affect the vehicle | Cleaned vehicle, component, part, and assignment data | Mark a vehicle as defective if the vehicle itself, one of its components, or one of its parts is defective. Keep the result unknown if there is no confirmed defect but some information is missing. |
| Vehicle, component, and part IDs should be unique | Cleaned vehicle, component, and part tables | Check that vehicle, component, and part IDs do not occur more than once after cleaning. |
| Each vehicle should have four components | Cleaned vehicle-component assignments | Count the component assignments for each vehicle and check that the result is always four. |
| Table joins must have the expected structure | Plant joins and the final vehicle-component-part trace | Use `validate` during merges to check whether the join is `many-to-one`, `one-to-one`, or `one-to-many`. |
| The final tables must have the expected size | `vehicle_quality_clean` and `quality_trace_clean` | Check the expected number of Type 11 vehicles, Type 21 vehicles, and rows in the detailed trace table. |
| Important IDs must not be missing in the final trace | `quality_trace_clean` | Check that `ID_Fahrzeug`, `ID_Komponente`, and `ID_Einzelteil` contain no missing values. |
| Some cleaning changes should be recorded | `.x`/`.y` conflicts, corrected manufacturer and plant numbers, duplicate component rows, and duplicate part rows | Save the source file, the type of cleaning, and the number of affected rows in `cleaning_log.csv`. |

# 5. Data Preparation and Integration

This section reads every relevant source file into a pandas DataFrame.

The work follows the data from vehicle down to part:

1. clean the plant master data,
2. clean the vehicle data,
3. reshape the vehicle-component mappings,
4. clean and combine the component files,
5. reshape the component-part mappings,
6. clean and combine the part files,
7. build the detailed trace table,
8. build the vehicle-level quality table.


## 5.1 Cleaning Configuration

The dictionaries below record information that is already known from the data exploration:

- which component belongs to which role (body, transmission, seats, engine),
- which part types belong to which component,
- which files use a comma instead of a semicolon,
- which TXT files need a special field and record separator.

The output folder `data/cleaned` is created here.


In [ ]:
CLEAN_DIRECTORY = DATA_DIRECTORY / "cleaned"
CLEAN_DIRECTORY.mkdir(exist_ok=True)

vehicle_data_paths = {
    "Type 11": vehicle_paths["Vehicle Type 11"],
    "Type 21": vehicle_paths["Vehicle Type 21"],
}

vehicle_mapping_paths = {
    "Type 11": vehicle_paths["Vehicle Type 11 mapping"],
    "Type 21": vehicle_paths["Vehicle Type 21 mapping"],
}

component_configuration = {
    "K4": "Karosserie",
    "K3AG1": "Schaltung",
    "K3SG1": "Schaltung",
    "K2LE1": "Sitze",
    "K2ST1": "Sitze",
    "K1BE1": "Motor",
    "K1DI1": "Motor",
    "K6": "Karosserie",
    "K3AG2": "Schaltung",
    "K3SG2": "Schaltung",
    "K2LE2": "Sitze",
    "K2ST2": "Sitze",
    "K1BE2": "Motor",
    "K1DI2": "Motor",
}

component_part_types = {
    "K4": ["T30", "T31", "T32"],
    "K3AG1": ["T21", "T24", "T25"],
    "K3SG1": ["T21", "T22", "T23"],
    "K2LE1": ["T11", "T14", "T15"],
    "K2ST1": ["T11", "T12", "T13"],
    "K1BE1": ["T01", "T02", "T03", "T04"],
    "K1DI1": ["T01", "T02", "T05", "T06"],
    "K6": ["T34", "T35", "T36", "T37"],
    "K3AG2": ["T21", "T24", "T27"],
    "K3SG2": ["T21", "T22", "T26"],
    "K2LE2": ["T16", "T19", "T20"],
    "K2ST2": ["T16", "T17", "T18"],
    "K1BE2": ["T01", "T02", "T07", "T08"],
    "K1DI2": ["T01", "T02", "T09", "T10"],
}

comma_separated_files = {
    "Fahrzeuge_OEM1_Typ11.csv",
    "Fahrzeuge_OEM2_Typ21.csv",
    "Komponente_K1BE1.csv",
    "Komponente_K1DI1.csv",
    "Komponente_K3AG1.csv",
    "Komponente_K3SG1.csv",
    "Komponente_K3SG2.csv",
    "Einzelteil_T05.csv",
    "Einzelteil_T06.csv",
    "Einzelteil_T08.csv",
    "Einzelteil_T19.csv",
    "Einzelteil_T25.csv",
    "Einzelteil_T30.csv",
    "Einzelteil_T37.csv",
}

special_text_formats = {
    "Komponente_K1DI2.txt": ("\\", "\t"),
    "Komponente_K2LE1.txt": ("II", "\x0b"),
    "Komponente_K2LE2.txt": ("\\", "\r"),
    "Komponente_K2ST1.txt": ("|", "\r"),
    "Komponente_K3AG2.txt": ("\\", "\r"),
    "Einzelteil_T01.txt": (" | | ", None),
    "Einzelteil_T02.txt": ("  ", "\t"),
    "Einzelteil_T03.txt": ("|", "\x0b"),
    "Einzelteil_T07.txt": ("\t", None),
    "Einzelteil_T09.txt": ("\\", "\x0b"),
    "Einzelteil_T11.txt": ("\t", "\x0c"),
    "Einzelteil_T16.txt": (" | | ", "\t"),
    "Einzelteil_T20.txt": (" | | ", None),
    "Einzelteil_T22.txt": ("\t", None),
    "Einzelteil_T24.txt": ("  ", "\x0c"),
    "Einzelteil_T27.txt": (" | | ", "\x07"),
    "Einzelteil_T31.txt": ("  ", "\x08"),
    "Einzelteil_T34.txt": (" | | ", None),
    "Einzelteil_T35.txt": ("\\", None),
    "Einzelteil_T36.txt": ("  ", None),
}

# Collects corrections and conflicts found during cleaning.
cleaning_log = []


## 5.2 Reading the Raw Source Files

Most files can be read with `pd.read_csv`. The previews showed, however, that several TXT files use unusual field separators and control characters instead of normal line breaks.

`read_special_text_file` reconstructs those files:

- if a record separator is known (for example a vertical tab), the file is split on that character
- if there is no record separator, a new row starts at a quoted numeric index (`"0"`, `"1"`, ...)
- each record is then split into columns with the field separator identified before

`mmap` is used because some of these TXT files are large. It lets us search the file without first copying the whole content into a Python string.

`read_source_file` is the wrapper function that reads in all files. Regular CSV files are read completely and special TXT files go through the dedicated reader function.


In [ ]:
def split_special_record(record, field_separator):
    """Split one TXT record into a list of field values.

    The files use different separators, so one split method is not enough.
    After splitting we only strip leftover spaces and quotes.
    """
    # cp1252 because the source files are Windows exports.
    text = record.decode("cp1252", errors="replace").strip()

    if field_separator == "II":
        # Fixed "II" separator, e.g. K2LE1.
        values = text.split("II")
    elif field_separator == " | | ":
        # Number of spaces around the pipes is not always the same.
        values = re.split(r"\s*\|\s*\|\s*", text)
    elif field_separator == "  ":
        # Two or more spaces = new field. A single space can still be inside a value.
        values = re.split(r" {2,}", text)
    else:
        # Backslash, pipe, tab, etc. csv.reader also handles quoted fields.
        values = next(
            csv.reader(
                [text],
                delimiter=field_separator,
                quotechar='"',
                skipinitialspace=True,
            )
        )

    return [value.strip().strip('"') for value in values]


def special_row_boundary(field_separator):
    """Build a regex that finds the start of a new row.

    Some TXT files have no line breaks at all, just one long stream.
    In those files a new row starts at a quoted index like "0", "1", "2", ...

    We cannot search for every quoted number though: the same pattern can also
    appear in the middle of a row. So we only keep a match if it is NOT sitting
    directly after a field separator.

    We also do not require an ID right after the index. In later .x / .y blocks
    the first ID field can be NA, so that would miss real row starts.
    """
    # Bytes patterns, because we search in the mmap.
    if field_separator == " | | ":
        separator_pattern = rb"\s*\|\s*\|\s*"
        not_after_separator = rb"(?<! \| \| )"
    elif field_separator == "  ":
        separator_pattern = rb" {2,}"
        not_after_separator = rb"(?<!  )"
    elif field_separator == "\t":
        separator_pattern = rb"\t"
        not_after_separator = rb"(?<!\t)"
    elif field_separator == "\\":
        separator_pattern = rb"\\"
        not_after_separator = rb"(?<!\\)"
    elif field_separator == "|":
        separator_pattern = rb"\|"
        not_after_separator = rb"(?<!\|)"
    else:
        encoded_separator = re.escape(field_separator.encode("cp1252"))
        separator_pattern = encoded_separator
        not_after_separator = rb"(?<!" + encoded_separator + rb")"

    return re.compile(not_after_separator + rb'(?="\d+"' + separator_pattern + rb")")


def extract_special_records(mapped_file, field_separator, record_separator):
    """Cut the mapped file into one byte-slice per record (header + data rows).

    Two cases:
    1) The file has a record separator (vertical tab, CR, ...): split there.
    2) It does not: search for row starts with special_row_boundary().
    """
    if record_separator is not None:
        separator = record_separator.encode("cp1252")
        start = 0

        # Walk through the file and cut at every separator.
        while True:
            end = mapped_file.find(separator, start)
            if end == -1:
                # Last piece (no separator after it anymore).
                if start < len(mapped_file):
                    yield mapped_file[start:]
                break
            yield mapped_file[start:end]
            start = end + len(separator)

        return

    # No record separator: the whole file is one stream.
    # Each regex match is the start of a data row. Everything before the
    # first match is the header.
    boundary = special_row_boundary(field_separator)
    previous_start = 0
    first_match = True

    for match in boundary.finditer(mapped_file):
        if first_match:
            yield mapped_file[:match.start()]
            first_match = False
        else:
            yield mapped_file[previous_start:match.start()]
        previous_start = match.start()

    if first_match:
        raise ValueError("No data-row boundary could be found.")

    # Last row goes until the end of the file.
    yield mapped_file[previous_start:]


def read_special_text_file(file_path):
    """Read one non-standard TXT file and return it as a DataFrame.

    Steps:
    1) look up the separators for this filename
    2) split the file into records
    3) split each record into fields
    4) check that every row has the same number of columns
    """
    field_separator, record_separator = special_text_formats[file_path.name]

    # mmap to view the file as bytes without copying it into one huge string.
    with file_path.open("rb") as source_file:
        with mmap.mmap(
            source_file.fileno(),
            length=0,
            access=mmap.ACCESS_READ,
        ) as mapped_file:
            records = extract_special_records(
                mapped_file,
                field_separator,
                record_separator,
            )
            # First record is column names, the rest are data rows.
            header = split_special_record(next(records), field_separator)
            rows = [
                split_special_record(record, field_separator)
                for record in records
                if record.strip()
            ]

    # Some files put a row index in the data, but not in the header.
    # Then the first data row is one field longer -> add a dummy column name.
    if rows and len(rows[0]) == len(header) + 1:
        header = ["_Zeilenindex", *header]

    # If a later row has a different width, the separators were probably wrong.
    invalid_rows = [
        row_number
        for row_number, row in enumerate(rows, start=1)
        if len(row) != len(header)
    ]
    if invalid_rows:
        raise ValueError(
            f"Unexpected field count in {file_path.name}; "
            f"first invalid data row: {invalid_rows[0]}"
        )

    return pd.DataFrame(rows, columns=header, dtype="string")


def read_source_file(file_path):
    """Read one complete source file into a DataFrame."""
    if file_path.name in special_text_formats:
        return read_special_text_file(file_path)

    separator = "," if file_path.name in comma_separated_files else ";"
    return pd.read_csv(
        file_path,
        sep=separator,
        encoding="cp1252",
        encoding_errors="replace",
        dtype="string",
        keep_default_na=False,
        low_memory=False,
    )


## 5.3 Recurring Cleaning Steps

The functions below each solve problems that appeared in many files (as seen in the aforementioned table):

- **standardize_raw_values:** strip whitespace and replace empty strings with a real missing value,
- **remove_export_columns:** drop duplicate row counters,
- **combine_duplicate_columns:** merge regular, `.x`, and `.y` columns into one column,
- **standardize_dates:** convert both normal dates and origin-based day counts to `YYYY-MM-DD`,
- **correct_keys_from_id:** take manufacturer and plant number from the structured ID if the explicit columns disagree.

Every correction is written to `cleaning_log`.


In [ ]:
def standardize_raw_values(data):
    """Make values comparable: trim spaces and replace missing-value text with pd.NA.

    The files mix empty strings, 'NA', 'NULL', etc. If we leave them as text,
    later joins would treat them as real values.
    """
    cleaned = data.copy()
    cleaned.columns = [str(column).strip() for column in cleaned.columns]

    for column in cleaned.columns:
        cleaned[column] = cleaned[column].astype("string").str.strip()

    missing_markers = {
        "": pd.NA,
        "NA": pd.NA,
        "N/A": pd.NA,
        "NULL": pd.NA,
        "null": pd.NA,
    }
    return cleaned.replace(missing_markers)


def remove_export_columns(data):
    """Drop duplicate row counters."""
    technical_columns = [
        column
        for column in data.columns
        if column in {"X", "X1", "_Zeilenindex"}
        or column.startswith("Unnamed:")
        or column == ""
    ]
    return data.drop(columns=technical_columns, errors="ignore")


def combine_duplicate_columns(data, source_file):
    """Merge columns that exist as name, name.x and name.y into one column.

    Some exports contain the same attribute twice as a .x and a .y version.
    We keep the first non-missing value. If two filled versions disagree,
    we keep the first one and log the conflict in cleaning_log.
    """
    # Unique base names.
    base_columns = []
    for column in data.columns:
        base_column = re.sub(r"\.[xy]$", "", column)
        if base_column not in base_columns:
            base_columns.append(base_column)

    combined = pd.DataFrame(index=data.index)
    conflict_count = 0

    for base_column in base_columns:
        # Take the column without a suffix, then .x, then .y.
        candidates = [
            column
            for column in [base_column, f"{base_column}.x", f"{base_column}.y"]
            if column in data.columns
        ]

        combined_values = pd.Series(pd.NA, index=data.index, dtype="string")
        for candidate in candidates:
            candidate_values = data[candidate].astype("string")
            # It's a conflict when both sides have a different value.
            conflicts = (
                combined_values.notna()
                & candidate_values.notna()
                & combined_values.ne(candidate_values)
            )
            conflict_count += int(conflicts.fillna(False).sum())
            # Empty cells get filled, filled cells stay the same.
            combined_values = combined_values.combine_first(candidate_values)

        combined[base_column] = combined_values

    if conflict_count > 0:
        cleaning_log.append(
            {
                "Quelldatei": source_file,
                "Prüfung": "Conflicting .x/.y values",
                "Anzahl": conflict_count,
            }
        )

    return combined


In [ ]:
def standardize_dates(data):
    """Turn all production / defect dates into YYYY-MM-DD.

    Two formats show up:
    - a normal date column (Produktionsdatum)
    - a day count since an origin date (Produktionsdatum_Origin_01011970)

    If both exist, we keep the normal date and only fill gaps from the day count.
    """
    cleaned = data.copy()
    origin_value_column = "Produktionsdatum_Origin_01011970"

    if origin_value_column in cleaned.columns:
        # Day count needs a start date. Some files store it in 'origin',
        # otherwise the column name already says 01.01.1970.
        if "origin" in cleaned.columns:
            origin_dates = pd.to_datetime(
                cleaned["origin"],
                format="%d-%m-%Y",
                errors="coerce",
            )
        else:
            origin_dates = pd.Series(
                pd.Timestamp("1970-01-01"),
                index=cleaned.index,
            )

        day_values = pd.to_numeric(
            cleaned[origin_value_column],
            errors="coerce",
        )
        converted_dates = origin_dates + pd.to_timedelta(day_values, unit="D")

        if "Produktionsdatum" in cleaned.columns:
            existing_dates = pd.to_datetime(
                cleaned["Produktionsdatum"],
                format="mixed",
                errors="coerce",
            )
        else:
            existing_dates = pd.Series(pd.NaT, index=cleaned.index)

        # Take cleaned date column.
        cleaned["Produktionsdatum"] = existing_dates.fillna(converted_dates)
        cleaned = cleaned.drop(
            columns=[origin_value_column, "origin"],
            errors="ignore",
        )

    # One common string format for all date columns.
    for date_column in ["Produktionsdatum", "Fehlerhaft_Datum"]:
        if date_column in cleaned.columns:
            parsed_dates = pd.to_datetime(
                cleaned[date_column],
                format="mixed",
                errors="coerce",
            )
            cleaned[date_column] = parsed_dates.dt.strftime("%Y-%m-%d")

    return cleaned


def correct_keys_from_id(data, id_column, source_file):
    """If Herstellernummer / Werksnummer differs from ID, take the ID.

    We join plants later on Werksnummer, so a wrong plant number would lead to errors in merging. 
    Every overwrite is written to cleaning_log.
    """
    cleaned = data.copy()
    # Group 1 = manufacturer, group 2 = plant. Everything is separated by '-'.
    id_keys = cleaned[id_column].astype("string").str.extract(
        r"^[^-]+-([^-]+)-([^-]+)-"
    )
    id_keys.columns = ["Herstellernummer_ID", "Werksnummer_ID"]

    key_pairs = [
        ("Herstellernummer", "Herstellernummer_ID"),
        ("Werksnummer", "Werksnummer_ID"),
    ]

    for target_column, id_key_column in key_pairs:
        if target_column not in cleaned.columns:
            cleaned[target_column] = pd.NA

        # Remove ".0" leftover in Werk IDs.
        current_values = (
            cleaned[target_column]
            .astype("string")
            .str.replace(r"\.0$", "", regex=True)
        )
        id_values = id_keys[id_key_column].astype("string")
        corrections = (
            id_values.notna()
            & current_values.fillna("").ne(id_values)
        )
        correction_count = int(corrections.sum())

        if correction_count > 0:
            cleaning_log.append(
                {
                    "Quelldatei": source_file,
                    "Prüfung": f"Corrected {target_column} from ID",
                    "Anzahl": correction_count,
                }
            )

        # Overwrite mismatches with the ID segment, then fill remaining gaps.
        cleaned[target_column] = current_values.mask(corrections, id_values)
        cleaned[target_column] = cleaned[target_column].fillna(id_values)

    return cleaned


## 5.4 Plant Data

The OEM, Tier-1, and Tier-2 plant files are cleaned and combined into one common plant table. This table is later used to add location information to vehicles, components, and parts.

- Clean missing values and remove empty columns and incomplete rows.
- Standardise plant numbers, postal codes, and coordinates.
- Add the supply-chain level and source file.
- Combine all plant data into `plants_clean`.

In [ ]:
plant_source_files = {
    "OEM": plant_paths["OEM plants"],
    "Tier 1": plant_paths["Tier 1 plants"],
    "Tier 2": plant_paths["Tier 2 plants"],
}

plant_frames = []

for plant_level, file_path in plant_source_files.items():
    cleaned_plant_data = standardize_raw_values(read_source_file(file_path))
    cleaned_plant_data = cleaned_plant_data.dropna(axis=1, how="all")
    # One plant file uses a wrong spelling of Längengrad because of the Umlaut.
    cleaned_plant_data.columns = [
        "Längengrad" if "ngengrad" in column else column
        for column in cleaned_plant_data.columns
    ]
    cleaned_plant_data = cleaned_plant_data.dropna(subset=["Werk", "ORT", "PLZ"])
    cleaned_plant_data["Werksebene"] = plant_level
    cleaned_plant_data["Werksnummer"] = (
        cleaned_plant_data["Werk"].astype("string").str.replace(r"^O", "", regex=True)
    )
    cleaned_plant_data["PLZ"] = (
        cleaned_plant_data["PLZ"].str.replace(r"\.0$", "", regex=True).str.zfill(5)
    )
    cleaned_plant_data["Breitengrad"] = pd.to_numeric(
        cleaned_plant_data["Breitengrad"],
        errors="coerce",
    )
    cleaned_plant_data["Längengrad"] = pd.to_numeric(
        cleaned_plant_data["Längengrad"],
        errors="coerce",
    )
    cleaned_plant_data["Quelldatei"] = file_path.name
    plant_frames.append(cleaned_plant_data)

plants_clean = pd.concat(plant_frames, ignore_index=True)

PLANT_COLUMNS = ["Werksnummer", "Werk", "PLZ", "ORT", "Breitengrad", "Längengrad"]


def plants_by_level(level):
    """Unique plants for one supply-chain level, ready to join."""
    return (
        plants_clean.loc[plants_clean["Werksebene"] == level, PLANT_COLUMNS]
        .drop_duplicates("Werksnummer")
    )


plants_clean.groupby("Werksebene").size().rename("Anzahl_Werke").to_frame()


## 5.5 Vehicle Data

The vehicle files for Type 11 and Type 21 are cleaned in the same way and combined into one table. The related OEM plant information is also added to each vehicle.

- Remove technical and duplicated columns.
- Standardise dates, IDs, and numeric values.
- Add the vehicle type, OEM, and source file.
- Join the OEM plant data.
- Combine both vehicle types into `vehicles_clean`.

In [ ]:
vehicle_raw_data = {
    vehicle_type: read_source_file(file_path)
    for vehicle_type, file_path in vehicle_data_paths.items()
}

oem_plants = plants_by_level("OEM")
vehicle_frames = []

for vehicle_type, raw_vehicle_data in vehicle_raw_data.items():
    source_file = vehicle_data_paths[vehicle_type].name
    cleaned_vehicle_data = standardize_raw_values(raw_vehicle_data)
    cleaned_vehicle_data = remove_export_columns(cleaned_vehicle_data)
    cleaned_vehicle_data = combine_duplicate_columns(
        cleaned_vehicle_data,
        source_file,
    )
    cleaned_vehicle_data = standardize_dates(cleaned_vehicle_data)
    cleaned_vehicle_data = correct_keys_from_id(
        cleaned_vehicle_data,
        "ID_Fahrzeug",
        source_file,
    )
    cleaned_vehicle_data["Fehlerhaft"] = pd.to_numeric(
        cleaned_vehicle_data["Fehlerhaft"],
        errors="coerce",
    ).astype("Int64")
    cleaned_vehicle_data["Fehlerhaft_Fahrleistung"] = pd.to_numeric(
        cleaned_vehicle_data["Fehlerhaft_Fahrleistung"],
        errors="coerce",
    )
    cleaned_vehicle_data["Fahrzeugtyp"] = vehicle_type
    cleaned_vehicle_data["OEM"] = (
        "OEM1" if vehicle_type == "Type 11" else "OEM2"
    )
    cleaned_vehicle_data["Quelldatei"] = source_file
    cleaned_vehicle_data = cleaned_vehicle_data.merge(
        oem_plants,
        on="Werksnummer",
        how="left",
        validate="many_to_one",
    )
    vehicle_frames.append(cleaned_vehicle_data)

vehicles_clean = pd.concat(vehicle_frames, ignore_index=True)
vehicles_clean = vehicles_clean[
    [
        "ID_Fahrzeug",
        "Fahrzeugtyp",
        "OEM",
        "Produktionsdatum",
        "Herstellernummer",
        "Werksnummer",
        "Fehlerhaft",
        "Fehlerhaft_Datum",
        "Fehlerhaft_Fahrleistung",
        "Werk",
        "PLZ",
        "ORT",
        "Breitengrad",
        "Längengrad",
        "Quelldatei",
    ]
]

del vehicle_raw_data, vehicle_frames

vehicles_clean.groupby("Fahrzeugtyp").size().rename("Anzahl_Fahrzeuge").to_frame()


## 5.6 Vehicle-Component Mapping

The mapping files show which body, transmission, seats, and engine are installed in each vehicle. The original wide format is changed into a long format with one vehicle-component assignment per row.

- Clean both vehicle-component mapping files.
- Transform the four component columns into separate rows.
- Add the component role and type.
- Combine the results into `vehicle_components_clean`.

In [ ]:
vehicle_component_raw_data = {
    vehicle_type: read_source_file(file_path)
    for vehicle_type, file_path in vehicle_mapping_paths.items()
}

vehicle_component_frames = []
component_role_names = {
    "ID_Karosserie": "Karosserie",
    "ID_Schaltung": "Schaltung",
    "ID_Sitze": "Sitze",
    "ID_Motor": "Motor",
}

for vehicle_type, raw_mapping_data in vehicle_component_raw_data.items():
    cleaned_mapping_data = standardize_raw_values(raw_mapping_data)
    cleaned_mapping_data = remove_export_columns(cleaned_mapping_data)
    # One vehicle has four installed components, stored in four wide columns.
    long_mapping_data = cleaned_mapping_data.melt(
        id_vars="ID_Fahrzeug",
        value_vars=list(component_role_names),
        var_name="Komponentenfeld",
        value_name="ID_Komponente",
    )
    long_mapping_data["Komponentenrolle"] = (
        long_mapping_data["Komponentenfeld"].map(component_role_names)
    )
    # Component type is the prefix of the ID, e.g. K4-... becomes K4.
    long_mapping_data["Komponententyp"] = (
        long_mapping_data["ID_Komponente"]
        .astype("string")
        .str.extract(r"^([A-Za-z0-9]+)-", expand=False)
    )
    long_mapping_data["Fahrzeugtyp"] = vehicle_type
    long_mapping_data["Quelldatei"] = vehicle_mapping_paths[vehicle_type].name
    vehicle_component_frames.append(long_mapping_data)

vehicle_components_clean = pd.concat(
    vehicle_component_frames,
    ignore_index=True,
)
vehicle_components_clean = vehicle_components_clean[
    [
        "ID_Fahrzeug",
        "Fahrzeugtyp",
        "Komponentenrolle",
        "Komponententyp",
        "ID_Komponente",
        "Quelldatei",
    ]
]

del vehicle_component_raw_data, vehicle_component_frames

vehicle_components_clean.groupby(
    ["Fahrzeugtyp", "Komponentenrolle"]
).size().rename("Anzahl_Zuordnungen").to_frame()


## 5.7 Component Data

All 14 relevant component files are cleaned and combined into one component table. Each component is also connected to its Tier-1 production plant.

- Clean all component files.
- Standardise columns, dates, IDs, and numeric values.
- Combine rows with duplicate component IDs.
- Add the component type, role, and Tier-1 plant data.
- Combine all files into `components_clean`.

In [ ]:
tier_1_plants = plants_by_level("Tier 1")

component_frames = []

for component_type, component_role in component_configuration.items():
    source_file = component_paths[f"{component_type} production"]
    raw_component_data = read_source_file(source_file)
    cleaned_component_data = standardize_raw_values(raw_component_data)
    cleaned_component_data = remove_export_columns(cleaned_component_data)
    cleaned_component_data = combine_duplicate_columns(
        cleaned_component_data,
        source_file.name,
    )

    component_id_columns = [
        column
        for column in cleaned_component_data.columns
        if column.startswith("ID_")
    ]
    if len(component_id_columns) != 1:
        raise ValueError(
            f"Expected one component ID column in {source_file.name}."
        )

    component_id_column = component_id_columns[0]
    cleaned_component_data = standardize_dates(cleaned_component_data)
    cleaned_component_data = correct_keys_from_id(
        cleaned_component_data,
        component_id_column,
        source_file.name,
    )
    cleaned_component_data["Fehlerhaft"] = pd.to_numeric(
        cleaned_component_data["Fehlerhaft"],
        errors="coerce",
    ).astype("Int64")
    cleaned_component_data["Fehlerhaft_Fahrleistung"] = pd.to_numeric(
        cleaned_component_data["Fehlerhaft_Fahrleistung"],
        errors="coerce",
    )

    duplicate_component_rows = cleaned_component_data.duplicated(
        component_id_column,
        keep=False,
    )
    if duplicate_component_rows.any():
        cleaning_log.append(
            {
                "Quelldatei": source_file.name,
                "Prüfung": "Duplicate component rows consolidated",
                "Anzahl": int(duplicate_component_rows.sum()),
            }
        )
        # If the same component ID appears twice, keep a defect if any copy is defective.
        component_aggregation = {
            column: "max" if column == "Fehlerhaft" else "first"
            for column in cleaned_component_data.columns
            if column != component_id_column
        }
        cleaned_component_data = cleaned_component_data.groupby(
            component_id_column,
            as_index=False,
            dropna=False,
        ).agg(component_aggregation)

    cleaned_component_data = cleaned_component_data.rename(
        columns={component_id_column: "ID_Komponente"}
    )
    cleaned_component_data["Komponententyp"] = component_type
    cleaned_component_data["Komponentenrolle"] = component_role
    cleaned_component_data["Quelldatei"] = source_file.name
    cleaned_component_data = cleaned_component_data.merge(
        tier_1_plants,
        on="Werksnummer",
        how="left",
        validate="many_to_one",
    )
    component_frames.append(cleaned_component_data)

components_clean = pd.concat(component_frames, ignore_index=True)
components_clean = components_clean[
    [
        "ID_Komponente",
        "Komponententyp",
        "Komponentenrolle",
        "Produktionsdatum",
        "Herstellernummer",
        "Werksnummer",
        "Fehlerhaft",
        "Fehlerhaft_Datum",
        "Fehlerhaft_Fahrleistung",
        "Werk",
        "PLZ",
        "ORT",
        "Breitengrad",
        "Längengrad",
        "Quelldatei",
    ]
]

del component_frames, raw_component_data

components_clean.groupby("Komponententyp").size().rename(
    "Anzahl_Komponenten"
).to_frame()


## 5.8 Component-Part Mapping

The component-part files show which individual parts are installed in each component. They are transformed into one table with one component-part assignment per row.

- Clean the component-part mapping files.
- Select the expected part columns for each component type.
- Transform the part columns into separate rows.
- Combine the assignments into `component_parts_clean`.

In [ ]:
component_part_frames = []

for component_type, expected_part_types in component_part_types.items():
    source_file = component_paths[f"{component_type} mapping"]
    raw_component_part_data = read_source_file(source_file)
    cleaned_component_part_data = standardize_raw_values(
        raw_component_part_data
    )
    cleaned_component_part_data = remove_export_columns(
        cleaned_component_part_data
    )
    component_id_column = f"ID_{component_type}"

    if component_id_column not in cleaned_component_part_data.columns:
        raise ValueError(
            f"Missing {component_id_column} in {source_file.name}."
        )

    for part_type in expected_part_types:
        # Column names are not fully consistent (ID_T1 vs ID_T01).
        part_number = int(part_type[1:])
        possible_names = [f"ID_T{part_number:02d}", f"ID_T{part_number}"]
        part_columns = [
            column
            for column in dict.fromkeys(possible_names)
            if column in cleaned_component_part_data.columns
        ]

        if len(part_columns) != 1:
            raise ValueError(
                f"Could not identify {part_type} in {source_file.name}."
            )

        component_part_data = cleaned_component_part_data[
            [component_id_column, part_columns[0]]
        ].rename(
            columns={
                component_id_column: "ID_Komponente",
                part_columns[0]: "ID_Einzelteil",
            }
        )
        component_part_data["Komponententyp"] = component_type
        component_part_data["Einzelteiltyp"] = part_type
        component_part_data["Quelldatei"] = source_file.name
        component_part_frames.append(component_part_data)

component_parts_clean = pd.concat(
    component_part_frames,
    ignore_index=True,
)
component_parts_clean = component_parts_clean[
    [
        "ID_Komponente",
        "Komponententyp",
        "ID_Einzelteil",
        "Einzelteiltyp",
        "Quelldatei",
    ]
]

del component_part_frames, raw_component_part_data

component_parts_clean.groupby("Komponententyp").size().rename(
    "Anzahl_Einzelteilzuordnungen"
).to_frame()


## 5.9 Part Data

All relevant part files are cleaned and combined into one common table. Each part is also connected to its Tier-2 production plant.

- Clean all relevant part files.
- Standardise columns, dates, IDs, and numeric values.
- Combine rows with duplicate part IDs.
- Add the part type and Tier-2 plant data.
- Combine all files into `parts_clean`.

In [ ]:
tier_2_plants = plants_by_level("Tier 2")

part_frames = []

for part_type, source_file in part_paths.items():
    raw_part_data = read_source_file(source_file)
    cleaned_part_data = standardize_raw_values(raw_part_data)
    cleaned_part_data = remove_export_columns(cleaned_part_data)
    cleaned_part_data = combine_duplicate_columns(
        cleaned_part_data,
        source_file.name,
    )

    part_id_columns = [
        column
        for column in cleaned_part_data.columns
        if column.startswith("ID_")
    ]
    if len(part_id_columns) != 1:
        raise ValueError(
            f"Expected one part ID column in {source_file.name}."
        )

    part_id_column = part_id_columns[0]
    cleaned_part_data = standardize_dates(cleaned_part_data)
    cleaned_part_data = correct_keys_from_id(
        cleaned_part_data,
        part_id_column,
        source_file.name,
    )
    cleaned_part_data["Fehlerhaft"] = pd.to_numeric(
        cleaned_part_data["Fehlerhaft"],
        errors="coerce",
    ).astype("Int64")
    cleaned_part_data["Fehlerhaft_Fahrleistung"] = pd.to_numeric(
        cleaned_part_data["Fehlerhaft_Fahrleistung"],
        errors="coerce",
    )

    duplicate_part_rows = cleaned_part_data.duplicated(
        part_id_column,
        keep=False,
    )
    if duplicate_part_rows.any():
        cleaning_log.append(
            {
                "Quelldatei": source_file.name,
                "Prüfung": "Duplicate part rows consolidated",
                "Anzahl": int(duplicate_part_rows.sum()),
            }
        )
        # If the same part ID appears twice, keep a defect if any copy is defective.
        part_aggregation = {
            column: "max" if column == "Fehlerhaft" else "first"
            for column in cleaned_part_data.columns
            if column != part_id_column
        }
        cleaned_part_data = cleaned_part_data.groupby(
            part_id_column,
            as_index=False,
            dropna=False,
        ).agg(part_aggregation)

    cleaned_part_data = cleaned_part_data.rename(
        columns={part_id_column: "ID_Einzelteil"}
    )
    cleaned_part_data["Einzelteiltyp"] = part_type
    cleaned_part_data["Quelldatei"] = source_file.name
    cleaned_part_data = cleaned_part_data.merge(
        tier_2_plants,
        on="Werksnummer",
        how="left",
        validate="many_to_one",
    )
    part_frames.append(cleaned_part_data)

parts_clean = pd.concat(part_frames, ignore_index=True)
parts_clean = parts_clean[
    [
        "ID_Einzelteil",
        "Einzelteiltyp",
        "Produktionsdatum",
        "Herstellernummer",
        "Werksnummer",
        "Fehlerhaft",
        "Fehlerhaft_Datum",
        "Fehlerhaft_Fahrleistung",
        "Werk",
        "PLZ",
        "ORT",
        "Breitengrad",
        "Längengrad",
        "Quelldatei",
    ]
]

del part_frames, raw_part_data

parts_clean.groupby("Einzelteiltyp").size().rename(
    "Anzahl_Einzelteile"
).to_frame()


## 5.10 Intermediate Checks

Before the tables are joined, their basic structure is checked. This ensures that the main IDs are complete and unique and that every vehicle has the expected number of components.

- Show the size and missing keys of the six cleaned tables.
- Check that vehicle, component, and part IDs are unique.
- Check that every vehicle has four components.

In [ ]:
intermediate_table_summary = pd.DataFrame(
    [
        {
            "Table": "plants_clean",
            "Rows": len(plants_clean),
            "Main key": "Werksebene + Werksnummer",
            "Missing main key": int(plants_clean["Werksnummer"].isna().sum()),
        },
        {
            "Table": "vehicles_clean",
            "Rows": len(vehicles_clean),
            "Main key": "ID_Fahrzeug",
            "Missing main key": int(vehicles_clean["ID_Fahrzeug"].isna().sum()),
        },
        {
            "Table": "vehicle_components_clean",
            "Rows": len(vehicle_components_clean),
            "Main key": "ID_Fahrzeug + ID_Komponente",
            "Missing main key": int(
                vehicle_components_clean[
                    ["ID_Fahrzeug", "ID_Komponente"]
                ].isna().any(axis=1).sum()
            ),
        },
        {
            "Table": "components_clean",
            "Rows": len(components_clean),
            "Main key": "ID_Komponente",
            "Missing main key": int(components_clean["ID_Komponente"].isna().sum()),
        },
        {
            "Table": "component_parts_clean",
            "Rows": len(component_parts_clean),
            "Main key": "ID_Komponente + ID_Einzelteil",
            "Missing main key": int(
                component_parts_clean[
                    ["ID_Komponente", "ID_Einzelteil"]
                ].isna().any(axis=1).sum()
            ),
        },
        {
            "Table": "parts_clean",
            "Rows": len(parts_clean),
            "Main key": "ID_Einzelteil",
            "Missing main key": int(parts_clean["ID_Einzelteil"].isna().sum()),
        },
    ]
)

assert not vehicles_clean["ID_Fahrzeug"].duplicated().any()
assert not components_clean["ID_Komponente"].duplicated().any()
assert not parts_clean["ID_Einzelteil"].duplicated().any()
assert (
    vehicle_components_clean.groupby("ID_Fahrzeug").size().eq(4).all()
)

intermediate_table_summary


## 5.11 Detailed Quality Trace

The cleaned tables are joined to create the complete path from each vehicle to its components and individual parts. The result can later be used to investigate the exact source of a defect.

- Add suffixes to separate vehicle, component, and part columns.
- Join vehicles with their installed components.
- Add the related parts and their master data.
- Store the result in `quality_trace_clean`.

In [ ]:
LEVEL_COLUMNS = [
    "Produktionsdatum",
    "Herstellernummer",
    "Werksnummer",
    "Fehlerhaft",
    "Fehlerhaft_Datum",
    "Fehlerhaft_Fahrleistung",
    "Werk",
    "PLZ",
    "ORT",
    "Breitengrad",
    "Längengrad",
]


def add_level_suffix(data, suffix):
    """Add a suffix so vehicle, component, and part fields stay distinguishable."""
    return data.rename(columns={column: f"{column}_{suffix}" for column in LEVEL_COLUMNS})


vehicle_trace_data = add_level_suffix(vehicles_clean, "Fahrzeug").drop(columns="Quelldatei")
component_trace_data = add_level_suffix(components_clean, "Komponente").drop(columns="Quelldatei")
part_trace_data = add_level_suffix(parts_clean, "Einzelteil").drop(columns="Quelldatei")

quality_trace_clean = vehicle_components_clean.drop(columns="Quelldatei").merge(
    vehicle_trace_data,
    on=["ID_Fahrzeug", "Fahrzeugtyp"],
    how="left",
    validate="many_to_one",
)

quality_trace_clean = quality_trace_clean.merge(
    component_trace_data,
    on=["ID_Komponente", "Komponententyp", "Komponentenrolle"],
    how="left",
    validate="many_to_one",
)

quality_trace_clean = quality_trace_clean.merge(
    component_parts_clean.drop(columns="Quelldatei"),
    on=["ID_Komponente", "Komponententyp"],
    how="left",
    validate="one_to_many",
)

quality_trace_clean = quality_trace_clean.merge(
    part_trace_data,
    on=["ID_Einzelteil", "Einzelteiltyp"],
    how="left",
    validate="many_to_one",
)

quality_trace_clean.head()


## 5.12 Vehicle-Level Quality Table

The detailed trace contains several rows for each vehicle. Therefore, the defect information is grouped into one final quality result per vehicle.

- Calculate component defects for each vehicle.
- Calculate part defects for each vehicle.
- Combine them with the vehicle defect status.
- Create the overall three-state defect flag.
- Store one row per vehicle in `vehicle_quality_clean`.

In [ ]:
# Count each installed component once. The trace table would count the same
# component several times because it contains several parts.
installed_component_quality = vehicle_components_clean[
    ["ID_Fahrzeug", "ID_Komponente"]
].merge(
    components_clean[["ID_Komponente", "Fehlerhaft"]].rename(
        columns={"Fehlerhaft": "Fehlerhaft_Komponente_Roh"}
    ),
    on="ID_Komponente",
    how="left",
    validate="many_to_one",
)

installed_component_quality["Komponente_Ist_Fehlerhaft"] = pd.to_numeric(
    installed_component_quality["Fehlerhaft_Komponente_Roh"],
    errors="coerce",
).eq(1)
installed_component_quality["Komponentenstatus_Unbekannt"] = (
    installed_component_quality["Fehlerhaft_Komponente_Roh"].isna()
)

component_quality_by_vehicle = installed_component_quality.groupby(
    "ID_Fahrzeug",
    as_index=False,
).agg(
    Komponente_Hat_Fehler=("Komponente_Ist_Fehlerhaft", "max"),
    Komponentenstatus_Unbekannt=("Komponentenstatus_Unbekannt", "max"),
    Anzahl_Fehlerhafte_Komponenten=("Komponente_Ist_Fehlerhaft", "sum"),
)

part_quality_source = quality_trace_clean[
    ["ID_Fahrzeug", "Fehlerhaft_Einzelteil"]
].copy()
part_quality_source["Einzelteil_Ist_Fehlerhaft"] = pd.to_numeric(
    part_quality_source["Fehlerhaft_Einzelteil"],
    errors="coerce",
).eq(1)
part_quality_source["Einzelteilstatus_Unbekannt"] = (
    part_quality_source["Fehlerhaft_Einzelteil"].isna()
)

part_quality_by_vehicle = part_quality_source.groupby(
    "ID_Fahrzeug",
    as_index=False,
).agg(
    Einzelteil_Hat_Fehler=("Einzelteil_Ist_Fehlerhaft", "max"),
    Einzelteilstatus_Unbekannt=("Einzelteilstatus_Unbekannt", "max"),
    Anzahl_Fehlerhafte_Einzelteile=("Einzelteil_Ist_Fehlerhaft", "sum"),
)

vehicle_quality_clean = vehicles_clean.rename(
    columns={"Fehlerhaft": "Fehlerhaft_Fahrzeug"}
).merge(
    component_quality_by_vehicle,
    on="ID_Fahrzeug",
    how="left",
    validate="one_to_one",
).merge(
    part_quality_by_vehicle,
    on="ID_Fahrzeug",
    how="left",
    validate="one_to_one",
)


def three_state_flag(has_defect, has_unknown):
    """1 = confirmed defect, 0 = all OK, <NA> = incomplete but no confirmed defect."""
    flag = pd.Series(0, index=has_defect.index, dtype="Int64")
    flag = flag.mask(~has_defect & has_unknown, pd.NA)
    flag = flag.mask(has_defect, 1)
    return flag


vehicle_quality_clean["Fehlerhaft_Komponente"] = three_state_flag(
    vehicle_quality_clean["Komponente_Hat_Fehler"],
    vehicle_quality_clean["Komponentenstatus_Unbekannt"],
)
vehicle_quality_clean["Fehlerhaft_Einzelteil"] = three_state_flag(
    vehicle_quality_clean["Einzelteil_Hat_Fehler"],
    vehicle_quality_clean["Einzelteilstatus_Unbekannt"],
)

vehicle_defect = pd.to_numeric(vehicle_quality_clean["Fehlerhaft_Fahrzeug"], errors="coerce")
component_defect = pd.to_numeric(vehicle_quality_clean["Fehlerhaft_Komponente"], errors="coerce")
part_defect = pd.to_numeric(vehicle_quality_clean["Fehlerhaft_Einzelteil"], errors="coerce")

confirmed_defect = (
    vehicle_defect.eq(1) | component_defect.eq(1) | part_defect.eq(1)
).fillna(False)
unknown_status = (
    vehicle_defect.isna() | component_defect.isna() | part_defect.isna()
)

vehicle_quality_clean["Fehlerhaft_Gesamt"] = three_state_flag(
    confirmed_defect,
    unknown_status,
)
vehicle_quality_clean["Datenstatus"] = np.where(
    unknown_status,
    "Teilweise unbekannt",
    "Vollständig",
)

vehicle_quality_clean = vehicle_quality_clean.drop(
    columns=[
        "Komponente_Hat_Fehler",
        "Komponentenstatus_Unbekannt",
        "Einzelteil_Hat_Fehler",
        "Einzelteilstatus_Unbekannt",
    ]
)

vehicle_quality_clean.head()


## 5.13 Final Validation

The final tables are checked before they are exported. These checks ensure that the expected number of vehicles and trace rows is present and that important IDs are complete.

- Compare the actual and expected vehicle counts.
- Check that every vehicle ID is unique.
- Check the expected number of trace rows.
- Check that important IDs are not missing.

In [ ]:
expected_vehicle_counts = {
    "Type 11": 1_977_164,
    "Type 21": 512_354,
}
expected_trace_rows = (
    expected_vehicle_counts["Type 11"] * 13
    + expected_vehicle_counts["Type 21"] * 14
)

actual_vehicle_counts = (
    vehicle_quality_clean.groupby("Fahrzeugtyp").size().to_dict()
)
trace_rows_by_vehicle_type = (
    quality_trace_clean.groupby("Fahrzeugtyp").size().to_dict()
)

assert actual_vehicle_counts == expected_vehicle_counts
assert not vehicle_quality_clean["ID_Fahrzeug"].duplicated().any()
assert len(quality_trace_clean) == expected_trace_rows
assert quality_trace_clean["ID_Fahrzeug"].notna().all()
assert quality_trace_clean["ID_Komponente"].notna().all()
assert quality_trace_clean["ID_Einzelteil"].notna().all()

final_validation = pd.DataFrame(
    [
        {
            "Check": "Vehicle Type 11 rows",
            "Expected": expected_vehicle_counts["Type 11"],
            "Actual": actual_vehicle_counts["Type 11"],
            "Passed": True,
        },
        {
            "Check": "Vehicle Type 21 rows",
            "Expected": expected_vehicle_counts["Type 21"],
            "Actual": actual_vehicle_counts["Type 21"],
            "Passed": True,
        },
        {
            "Check": "Detailed trace rows",
            "Expected": expected_trace_rows,
            "Actual": len(quality_trace_clean),
            "Passed": True,
        },
        {
            "Check": "Unique vehicle IDs",
            "Expected": len(vehicle_quality_clean),
            "Actual": vehicle_quality_clean["ID_Fahrzeug"].nunique(),
            "Passed": True,
        },
    ]
)

final_validation


## 5.14 Export of Clean Data

The cleaned tables are written as UTF-8 CSV files in `data/cleaned`.

`vehicle_quality_clean` is the main table for the OEM audit decision: one row per vehicle, with plant and defect information. The detailed vehicle-component-part trace stays in memory for this cleaning session, but it is **not** exported. At more than 30 million rows it is too large for the submission.

Instead, two compact supplier tables are written for the later defect-source analysis:

- `component_plant_quality.csv`: defect counts by Tier-1 plant and component type
- `part_plant_quality.csv`: defect counts by Tier-2 plant and part type

- Export the six intermediate tables and `vehicle_quality_clean`.
- Export the two compact supplier quality tables.
- Export the cleaning log.
- Do not export `quality_trace_clean`.

In [ ]:
def plant_type_quality(table, type_column, extra_group_columns=None):
    """Count items and confirmed defects by production plant and type."""
    group_columns = ["Werk", "Werksnummer", "ORT", type_column]
    if extra_group_columns:
        group_columns = group_columns + extra_group_columns

    working = table.loc[
        :,
        group_columns + ["Fehlerhaft", "Breitengrad", "Längengrad"],
    ].copy()
    working["Ist_Fehlerhaft"] = pd.to_numeric(
        working["Fehlerhaft"], errors="coerce"
    ).eq(1)

    summary = working.groupby(group_columns, dropna=False, as_index=False).agg(
        Anzahl=("Ist_Fehlerhaft", "size"),
        Fehler=("Ist_Fehlerhaft", "sum"),
        Breitengrad=("Breitengrad", "first"),
        Längengrad=("Längengrad", "first"),
    )
    summary["Fehlerquote"] = summary["Fehler"] / summary["Anzahl"]
    return summary.sort_values("Fehlerquote", ascending=False).reset_index(drop=True)


component_plant_quality = plant_type_quality(
    components_clean,
    "Komponententyp",
    extra_group_columns=["Komponentenrolle"],
)
part_plant_quality = plant_type_quality(parts_clean, "Einzelteiltyp")

clean_tables = {
    "vehicles_clean.csv": vehicles_clean,
    "vehicle_components_clean.csv": vehicle_components_clean,
    "components_clean.csv": components_clean,
    "component_parts_clean.csv": component_parts_clean,
    "parts_clean.csv": parts_clean,
    "plants_clean.csv": plants_clean,
    "vehicle_quality_clean.csv": vehicle_quality_clean,
    "component_plant_quality.csv": component_plant_quality,
    "part_plant_quality.csv": part_plant_quality,
}

# Write UTF-8 CSV files so the clean tables can be reused without repeating the import.
for file_name, clean_table in clean_tables.items():
    clean_table.to_csv(
        CLEAN_DIRECTORY / file_name,
        index=False,
        encoding="utf-8",
    )

# The detailed trace is kept in memory only. Remove a previous export if present.
trace_export_path = CLEAN_DIRECTORY / "quality_trace_clean.csv"
if trace_export_path.exists():
    trace_export_path.unlink()

cleaning_log_table = pd.DataFrame(cleaning_log)
cleaning_log_table.to_csv(
    CLEAN_DIRECTORY / "cleaning_log.csv",
    index=False,
    encoding="utf-8",
)

export_summary = pd.DataFrame(
    [
        {
            "Table": file_name.removesuffix(".csv"),
            "Rows": len(clean_table),
            "Output file": (CLEAN_DIRECTORY / file_name).relative_to(
                PROJECT_ROOT
            ).as_posix(),
        }
        for file_name, clean_table in clean_tables.items()
    ]
)

export_summary


# 6. Decision Criteria

The audit question is answered at the **OEM vehicle plant**, not at a supplier plant. The analytical question is:

> In which OEM plant is the relative share of field-defective vehicles highest, where a vehicle is defective if the vehicle itself, an installed component, or an installed part is defective?

The ranking rules are fixed **before** looking at the plant table:

1. **Primary (process quality):** effective relative defect rate `Fehlerhaft_Gesamt == 1` among all produced vehicles, with a 95% Wald confidence interval \(1.96 \times \sqrt{p(1-p)/n}\).
2. **Secondary (customer impact):** absolute number of effectively defective vehicles.
3. **Definition test:** rank plants once with only `Fehlerhaft_Fahrzeug` and once with `Fehlerhaft_Gesamt`. If the two rankings disagree, the task definition (`Fehlerhaft_Gesamt`) is used.
4. **Fairness:** O11 and O12 are compared only within Type 11. O21 produces only Type 21, so a higher rate there is an audit priority, not a causal proof against the assembly process.
5. **Tie-break:** overlapping confidence intervals mean no material site difference. In that case volume and supplier insights are used, not extra decimal places.

Missing defect flags stay unknown. They do not count as defects and remain in the denominator.

Supplier plants are a **second layer**. They explain which component and part types fail often, and at which Tier-1 / Tier-2 plant they were produced. They do not replace the OEM ranking.


# 7. OEM Plant Comparison

The OEM ranking uses `vehicle_quality_clean`. If the cleaning pipeline has just been run, the table is taken from memory. Otherwise it is loaded from `data/cleaned/vehicle_quality_clean.csv`. The time series in 6.4 also uses a yearly role table (`oem_role_yearly.csv`).


In [ ]:
import math

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [ ]:
CITY_LABELS = {
    "NUERNBERG": "Nürnberg",
    "NURNBERG": "Nürnberg",
    "BONN": "Bonn",
    "GOETTINGEN": "Göttingen",
    "GOTTINGEN": "Göttingen",
    "GÖTTINGEN": "Göttingen",
    "MUENCHEN": "München",
    "WUERZBURG": "Würzburg",
    "FUERTH": "Fürth",
}

PLANT_COLORS = {
    "O11 - Nürnberg": "#137CBD",
    "O12 - Bonn": "#F29D49",
    "O21 - Göttingen": "#1F9D8A",
}

# Same plant colour everywhere. Dash is only needed when several plants
# share one role colour (e.g. Engine at O11 / O12 / O21).
PLANT_LINESTYLES = {
    "O11 - Nürnberg": "solid",
    "O12 - Bonn": "dash",
    "O21 - Göttingen": "dot",
}

ROLE_LABELS = {
    "Motor": "Engine",
    "Sitze": "Seats",
    "Schaltung": "Transmission",
    "Karosserie": "Body",
}

ROLE_COLORS = {
    "Engine": "#E74C3C",
    "Seats": "#7D3C98",
    "Transmission": "#1E8449",
    "Body": "#2E86AB",
}

ROLE_ORDER = ["Engine", "Seats", "Transmission", "Body"]

SOURCE_ORDER = [
    "Nur Fahrzeug",
    "Nur Komponente",
    "Nur Einzelteil",
    "Fahrzeug + Komponente",
    "Fahrzeug + Einzelteil",
    "Komponente + Einzelteil",
    "Fahrzeug + Komponente + Einzelteil",
]


def plant_label(werk, city):
    """Build a stable plant label such as 'O11 - Nürnberg'."""
    city_key = "" if pd.isna(city) else str(city).strip().upper()
    city_name = CITY_LABELS.get(city_key, str(city).title() if pd.notna(city) else "")
    return f"{werk} - {city_name}".strip(" -")


def is_confirmed_defect(series):
    """True only for confirmed defects. Unknown values stay False."""
    return pd.to_numeric(series, errors="coerce").eq(1)


def wald_interval(rate, n_obs):
    """Return the 95% Wald interval for a binomial rate."""
    standard_error = np.sqrt(rate * (1 - rate) / n_obs)
    lower = (rate - 1.96 * standard_error).clip(0, 1)
    upper = (rate + 1.96 * standard_error).clip(0, 1)
    return lower, upper


def two_proportion_z(count_a, n_a, count_b, n_b):
    """Rate gap in percentage points and two-proportion z score."""
    p_a = count_a / n_a
    p_b = count_b / n_b
    pooled = (count_a + count_b) / (n_a + n_b)
    standard_error = math.sqrt(pooled * (1 - pooled) * (1 / n_a + 1 / n_b))
    z_score = (p_a - p_b) / standard_error if standard_error else 0.0
    return (p_a - p_b) * 100, z_score


def defect_source(vehicle_flag, component_flag, part_flag):
    """Describe which of the three defect levels are active."""
    has_vehicle = is_confirmed_defect(vehicle_flag)
    has_component = is_confirmed_defect(component_flag)
    has_part = is_confirmed_defect(part_flag)
    source = np.full(len(vehicle_flag), "Kein Fehler", dtype=object)
    source = np.where(has_vehicle & ~has_component & ~has_part, "Nur Fahrzeug", source)
    source = np.where(~has_vehicle & has_component & ~has_part, "Nur Komponente", source)
    source = np.where(~has_vehicle & ~has_component & has_part, "Nur Einzelteil", source)
    source = np.where(has_vehicle & has_component & ~has_part, "Fahrzeug + Komponente", source)
    source = np.where(has_vehicle & ~has_component & has_part, "Fahrzeug + Einzelteil", source)
    source = np.where(~has_vehicle & has_component & has_part, "Komponente + Einzelteil", source)
    source = np.where(
        has_vehicle & has_component & has_part,
        "Fahrzeug + Komponente + Einzelteil",
        source,
    )
    return pd.Series(source, index=vehicle_flag.index)


def load_or_reuse(table_name, csv_name, usecols=None):
    """Use an in-memory table from cleaning, otherwise load its CSV."""
    if table_name in globals() and globals()[table_name] is not None:
        return globals()[table_name]
    csv_path = CLEAN_DIRECTORY / csv_name
    if not csv_path.exists():
        raise FileNotFoundError(
            f"{csv_name} was not found. Run the cleaning export in section 4.14 first."
        )
    return pd.read_csv(csv_path, usecols=usecols)


def style_figure(figure, height=430):
    """Apply a compact layout that stays readable in the notebook."""
    figure.update_layout(
        height=height,
        margin=dict(l=50, r=30, t=70, b=50),
        paper_bgcolor="white",
        plot_bgcolor="white",
        legend_title_text="",
    )
    figure.update_xaxes(showgrid=False)
    figure.update_yaxes(gridcolor="#EAF0F4", zeroline=False)
    return figure


def mix_with_white(color, amount=0.55):
    """Lighter hex colour, used for part nodes of a component role."""
    color = color.lstrip("#")
    rgb = [int(color[i : i + 2], 16) for i in (0, 2, 4)]
    mixed = [int(channel + (255 - channel) * amount) for channel in rgb]
    return "#{:02X}{:02X}{:02X}".format(*mixed)


def hex_to_rgba(color, alpha):
    """Convert #RRGGBB to an rgba() string."""
    color = color.lstrip("#")
    red, green, blue = (int(color[i : i + 2], 16) for i in (0, 2, 4))
    return f"rgba({red},{green},{blue},{alpha})"


def rate_axis(values, percent_points=False):
    """Y-axis range and tick step so close rates stay readable."""
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    lo = float(np.min(vals))
    hi = float(np.max(vals))
    span = hi - lo
    if percent_points:
        min_span = 0.5
        pad = max(span * 0.2, 0.08)
        steps = [0.1, 0.2, 0.5, 1.0, 2.0]
        tickformat = ".1f"
    else:
        min_span = 0.005
        pad = max(span * 0.2, 0.0008)
        steps = [0.001, 0.002, 0.005, 0.01, 0.02]
        tickformat = ".1%"
    if span < min_span:
        mid = (lo + hi) / 2.0
        lo = mid - min_span / 2.0
        hi = mid + min_span / 2.0
    else:
        lo = max(0.0, lo - pad)
        hi = hi + pad
    span = hi - lo
    dtick = steps[-1]
    for step in steps:
        if span / step <= 8:
            dtick = step
            break
    return dict(range=[lo, hi], dtick=dtick, tickformat=tickformat, ticks="outside")


## 7.1 Vehicle Quality Table

**What the table shows.** One row per Type 11 or Type 21 vehicle, with the vehicle, component and part flags and the combined `Fehlerhaft_Gesamt`. Confirmed defects are counted as 1; unknown values are not treated as defects.

**Why it matters.** This is the unit of analysis for the plant ranking. Sections 6.2–6.4 only aggregate these rows.


In [ ]:
if "CLEAN_DIRECTORY" not in globals():
    PROJECT_ROOT = Path.cwd().resolve()
    DATA_DIRECTORY = PROJECT_ROOT / "data"
    CLEAN_DIRECTORY = DATA_DIRECTORY / "cleaned"

VEHICLE_QUALITY_COLUMNS = [
    "ID_Fahrzeug",
    "Fahrzeugtyp",
    "OEM",
    "Produktionsdatum",
    "Werksnummer",
    "Fehlerhaft_Fahrzeug",
    "Fehlerhaft_Komponente",
    "Fehlerhaft_Einzelteil",
    "Fehlerhaft_Gesamt",
    "Werk",
    "ORT",
    "Breitengrad",
    "Längengrad",
]

analysis_vehicles = load_or_reuse(
    "vehicle_quality_clean",
    "vehicle_quality_clean.csv",
    usecols=VEHICLE_QUALITY_COLUMNS,
).copy()

analysis_vehicles["Werk_Label"] = [
    plant_label(werk, city)
    for werk, city in zip(analysis_vehicles["Werk"], analysis_vehicles["ORT"])
]
analysis_vehicles["Effektiver_Fehler"] = is_confirmed_defect(
    analysis_vehicles["Fehlerhaft_Gesamt"]
)
analysis_vehicles["Direkter_Fahrzeugfehler"] = is_confirmed_defect(
    analysis_vehicles["Fehlerhaft_Fahrzeug"]
)
analysis_vehicles["Fehlerquelle"] = defect_source(
    analysis_vehicles["Fehlerhaft_Fahrzeug"],
    analysis_vehicles["Fehlerhaft_Komponente"],
    analysis_vehicles["Fehlerhaft_Einzelteil"],
)
analysis_vehicles["Produktionsjahr"] = pd.to_datetime(
    analysis_vehicles["Produktionsdatum"], errors="coerce"
).dt.year.astype("Int64")

print(
    f"{len(analysis_vehicles):,} vehicles in the analysis table "
    f"across {analysis_vehicles['Werk_Label'].nunique()} OEM plants."
)
analysis_vehicles.head()


## 7.2 Absolute and Relative Rates by OEM Plant

The primary ranking uses the effective defect rate. The direct vehicle rate is kept next to it for the definition test. The table is sorted by effective rate.


In [ ]:
oem_plant_quality = (
    analysis_vehicles.groupby(["Werk_Label", "Werk", "Fahrzeugtyp"], dropna=False)
    .agg(
        Fahrzeuge=("ID_Fahrzeug", "size"),
        Effektive_Fehler=("Effektiver_Fehler", "sum"),
        Direkte_Fahrzeugfehler=("Direkter_Fahrzeugfehler", "sum"),
        Breitengrad=("Breitengrad", "first"),
        Längengrad=("Längengrad", "first"),
        ORT=("ORT", "first"),
    )
    .reset_index()
)

oem_plant_quality["Effektive_Quote"] = (
    oem_plant_quality["Effektive_Fehler"] / oem_plant_quality["Fahrzeuge"]
)
oem_plant_quality["Direkte_Quote"] = (
    oem_plant_quality["Direkte_Fahrzeugfehler"] / oem_plant_quality["Fahrzeuge"]
)
(
    oem_plant_quality["CI_unten"],
    oem_plant_quality["CI_oben"],
) = wald_interval(
    oem_plant_quality["Effektive_Quote"],
    oem_plant_quality["Fahrzeuge"],
)

oem_plant_quality = oem_plant_quality.sort_values(
    ["Effektive_Quote", "Effektive_Fehler"],
    ascending=False,
).reset_index(drop=True)

oem_plant_quality.to_csv(
    CLEAN_DIRECTORY / "oem_plant_quality.csv",
    index=False,
    encoding="utf-8",
)

oem_plant_quality


In [ ]:
first_plant = oem_plant_quality.iloc[0]
second_plant = oem_plant_quality.iloc[1]
type11_plants = oem_plant_quality.loc[
    oem_plant_quality["Fahrzeugtyp"] == "Type 11"
].sort_values("Effektive_Quote", ascending=False)

effective_gap_pp, effective_z = two_proportion_z(
    first_plant["Effektive_Fehler"],
    first_plant["Fahrzeuge"],
    second_plant["Effektive_Fehler"],
    second_plant["Fahrzeuge"],
)

direct_ranking = oem_plant_quality.sort_values(
    "Direkte_Quote", ascending=False
).reset_index(drop=True)

if len(type11_plants) >= 2:
    type11_gap_pp, type11_z = two_proportion_z(
        type11_plants.iloc[0]["Effektive_Fehler"],
        type11_plants.iloc[0]["Fahrzeuge"],
        type11_plants.iloc[1]["Effektive_Fehler"],
        type11_plants.iloc[1]["Fahrzeuge"],
    )
    type11_overlap = (
        type11_plants.iloc[0]["CI_unten"] <= type11_plants.iloc[1]["CI_oben"]
        and type11_plants.iloc[1]["CI_unten"] <= type11_plants.iloc[0]["CI_oben"]
    )
else:
    type11_gap_pp, type11_z, type11_overlap = np.nan, np.nan, False

comparison_checks = pd.DataFrame(
    [
        {
            "Check": "Highest effective rate",
            "Result": first_plant["Werk_Label"],
        },
        {
            "Check": "Highest direct vehicle rate",
            "Result": direct_ranking.iloc[0]["Werk_Label"],
        },
        {
            "Check": "Definition test changes the ranking",
            "Result": first_plant["Werk_Label"] != direct_ranking.iloc[0]["Werk_Label"],
        },
        {
            "Check": f"Effective gap {first_plant['Werk_Label']} vs {second_plant['Werk_Label']}",
            "Result": f"{effective_gap_pp:.3f} pp (z = {effective_z:.1f})",
        },
        {
            "Check": "Type 11 effective gap O11 vs O12",
            "Result": f"{type11_gap_pp:.3f} pp (z = {type11_z:.1f})",
        },
        {
            "Check": "Type 11 confidence intervals overlap",
            "Result": bool(type11_overlap),
        },
    ]
)
comparison_checks


**What the chart shows.** Left: how many vehicles are effectively defective (customer volume). Right: the effective defect rate with a 95% interval (process quality). That right-hand rate is the audit criterion from Section 6.

**Why it matters.** Göttingen (O21) leads on the relative rate. Nürnberg has more defective vehicles in absolute terms, but it also builds more Type 11 cars. For a process audit the relative rate comes first, so O21 is the plant to visit first.


In [ ]:
plant_colors = [
    PLANT_COLORS.get(label, "#137CBD") for label in oem_plant_quality["Werk_Label"]
]

overview_figure = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Defective vehicles", "Effective defect rate"),
    horizontal_spacing=0.16,
)
overview_figure.add_trace(
    go.Bar(
        x=oem_plant_quality["Werk_Label"],
        y=oem_plant_quality["Effektive_Fehler"],
        marker_color=plant_colors,
        text=oem_plant_quality["Effektive_Fehler"].map(lambda value: f"{value:,.0f}"),
        textposition="outside",
        customdata=oem_plant_quality[["Fahrzeuge", "Fahrzeugtyp"]],
        hovertemplate=(
            "<b>%{x}</b><br>Defective: %{y:,.0f}"
            "<br>Vehicles: %{customdata[0]:,.0f}"
            "<br>Type: %{customdata[1]}<extra></extra>"
        ),
        showlegend=False,
    ),
    row=1,
    col=1,
)
overview_figure.add_trace(
    go.Bar(
        x=oem_plant_quality["Werk_Label"],
        y=oem_plant_quality["Effektive_Quote"] * 100,
        marker_color=plant_colors,
        text=(oem_plant_quality["Effektive_Quote"] * 100).map(
            lambda value: f"{value:.2f} %"
        ),
        textposition="outside",
        error_y=dict(
            type="data",
            symmetric=False,
            array=(oem_plant_quality["CI_oben"] - oem_plant_quality["Effektive_Quote"])
            * 100,
            arrayminus=(oem_plant_quality["Effektive_Quote"] - oem_plant_quality["CI_unten"])
            * 100,
            color="#123B5D",
            thickness=1.2,
        ),
        hovertemplate="<b>%{x}</b><br>Effective rate: %{y:.3f} %<extra></extra>",
        showlegend=False,
    ),
    row=1,
    col=2,
)
overview_figure.update_yaxes(title_text="Vehicles", row=1, col=1)
overview_figure.update_yaxes(
    title_text="Defect rate [%]",
    row=1,
    col=2,
    **rate_axis(oem_plant_quality["Effektive_Quote"] * 100, percent_points=True),
)
overview_figure.update_layout(title="Defective vehicles and effective rate by OEM plant")
style_figure(overview_figure)


**What the chart shows.** Light bars: only `Fehlerhaft_Fahrzeug`. Dark bars: a vehicle counts as defective if the vehicle, an installed component, or an installed part is defective (the task definition).

**Why it matters.** The ranking can change if only the vehicle file is used. The dark bars are the ones that decide the audit.


In [ ]:
definition_long = oem_plant_quality.melt(
    id_vars=["Werk_Label"],
    value_vars=["Direkte_Quote", "Effektive_Quote"],
    var_name="Definition",
    value_name="Quote",
)
definition_long["Definition"] = definition_long["Definition"].map(
    {
        "Direkte_Quote": "Vehicle flag only",
        "Effektive_Quote": "Vehicle, component, or part",
    }
)
definition_long["Quote_pct"] = definition_long["Quote"] * 100

definition_figure = px.bar(
    definition_long,
    x="Werk_Label",
    y="Quote_pct",
    color="Definition",
    barmode="group",
    color_discrete_sequence=["#A8D5F2", "#137CBD"],
    title="Defect rate by definition",
    labels={"Werk_Label": "OEM plant", "Quote_pct": "Defect rate [%]"},
)
definition_figure.update_traces(
    hovertemplate="<b>%{x}</b><br>%{fullData.name}: %{y:.3f} %<extra></extra>"
)
style_figure(definition_figure)


## 7.3 Defect Source at Vehicle Level

**What the chart shows.** Among vehicles that are effectively defective, which of the three flags (vehicle / component / part) are on, including combinations.

**Why it matters.** The effective rate is high because many vehicles fail through an installed component or part, not only through the vehicle flag. The OEM visit therefore cannot stop at final assembly.


In [ ]:
source_counts = (
    analysis_vehicles.loc[analysis_vehicles["Fehlerquelle"] != "Kein Fehler"]
    .groupby(["Werk_Label", "Fehlerquelle"], dropna=False)
    .size()
    .rename("Fahrzeuge")
    .reset_index()
)
source_counts["Anteil"] = source_counts["Fahrzeuge"] / source_counts.groupby(
    "Werk_Label"
)["Fahrzeuge"].transform("sum")

source_figure = px.bar(
    source_counts,
    x="Werk_Label",
    y="Anteil",
    color="Fehlerquelle",
    category_orders={"Fehlerquelle": SOURCE_ORDER},
    title="Defect source of defective vehicles",
    labels={
        "Werk_Label": "OEM plant",
        "Anteil": "Share of defective vehicles",
        "Fehlerquelle": "Defect source",
    },
    color_discrete_sequence=[
        "#A8D5F2",
        "#67B4DF",
        "#137CBD",
        "#8FD5CA",
        "#50B8A6",
        "#F6BE83",
        "#D95C59",
    ],
)
source_figure.update_traces(
    hovertemplate="%{fullData.name}<br>Share: %{y:.1%}<extra></extra>"
)
source_figure.update_yaxes(tickformat=".0%")
style_figure(source_figure, height=470)


## 7.4 Defect Rates over Time

**What the chart shows.** Use the dropdown to switch between the overall effective vehicle rate and the defect rate of installed Engine, Seats, Transmission and Body components, by OEM plant and production year. Variants of the same role are pooled (e.g. K1BE1 and K1DI1 as Engine). Plant colours stay on the overall series; role colours match Section 8. Dash pattern is the plant (solid Nürnberg, dashed Bonn, dotted Göttingen).

**Why it matters.** A single bad year should not decide the audit. The overall series stays high across years, and Engine sits clearly above Body. That is hard to explain with final assembly alone.


In [ ]:
yearly_quality = (
    analysis_vehicles.dropna(subset=["Produktionsjahr"])
    .groupby(["Produktionsjahr", "Werk_Label", "Fahrzeugtyp"], dropna=False)
    .agg(
        Fahrzeuge=("ID_Fahrzeug", "size"),
        Effektive_Fehler=("Effektiver_Fehler", "sum"),
    )
    .reset_index()
)
yearly_quality["Effektive_Quote"] = (
    yearly_quality["Effektive_Fehler"] / yearly_quality["Fahrzeuge"]
)

# Role rates need the vehicle-component mapping. The CSV is large, so the
# result is written once and reused on the next run.
role_yearly_path = CLEAN_DIRECTORY / "oem_role_yearly.csv"

if role_yearly_path.exists():
    yearly_role = pd.read_csv(role_yearly_path)
else:
    print("Building yearly role table (first run only)...")
    vehicle_keys = analysis_vehicles.dropna(subset=["Produktionsjahr"])[
        ["ID_Fahrzeug", "Werk_Label", "Produktionsjahr"]
    ]
    installed = load_or_reuse(
        "vehicle_components_clean",
        "vehicle_components_clean.csv",
        usecols=["ID_Fahrzeug", "ID_Komponente", "Komponentenrolle"],
    )
    installed = installed.loc[:, ["ID_Fahrzeug", "ID_Komponente", "Komponentenrolle"]]
    component_flags = load_or_reuse(
        "components_clean",
        "components_clean.csv",
        usecols=["ID_Komponente", "Fehlerhaft"],
    )
    component_flags = component_flags.drop_duplicates("ID_Komponente")
    installed = installed.merge(component_flags, on="ID_Komponente", how="left")
    installed = installed.merge(vehicle_keys, on="ID_Fahrzeug", how="inner")
    installed["Ist_Fehlerhaft"] = is_confirmed_defect(installed["Fehlerhaft"])
    installed["Role"] = installed["Komponentenrolle"].map(ROLE_LABELS)

    yearly_role = (
        installed.groupby(["Produktionsjahr", "Werk_Label", "Role"], dropna=False)
        .agg(Anzahl=("Ist_Fehlerhaft", "size"), Fehler=("Ist_Fehlerhaft", "sum"))
        .reset_index()
    )
    yearly_role["Quote"] = yearly_role["Fehler"] / yearly_role["Anzahl"]
    yearly_role.to_csv(role_yearly_path, index=False, encoding="utf-8")
    del installed, component_flags, vehicle_keys

plant_order = [plant for plant in PLANT_COLORS if plant in set(yearly_quality["Werk_Label"])]

trend_figure = go.Figure()
trace_views = []

for plant in plant_order:
    sub = yearly_quality.loc[yearly_quality["Werk_Label"] == plant].sort_values(
        "Produktionsjahr"
    )
    trend_figure.add_trace(
        go.Scatter(
            x=sub["Produktionsjahr"],
            y=sub["Effektive_Quote"],
            mode="lines+markers",
            name=f"{plant} · Overall",
            line=dict(color=PLANT_COLORS[plant], width=3),
            marker=dict(size=8, color=PLANT_COLORS[plant]),
            customdata=np.stack(
                [sub["Fahrzeuge"], sub["Effektive_Fehler"], sub["Fahrzeugtyp"]],
                axis=1,
            ),
            hovertemplate=(
                "<b>%{fullData.name}</b><br>Year: %{x:.0f}"
                "<br>Rate: %{y:.1%}"
                "<br>Defective: %{customdata[1]:,.0f}"
                "<br>Vehicles: %{customdata[0]:,.0f}<extra></extra>"
            ),
            visible=True,
        )
    )
    trace_views.append("Overall")

for role in ROLE_ORDER:
    for plant in plant_order:
        sub = yearly_role.loc[
            (yearly_role["Role"] == role) & (yearly_role["Werk_Label"] == plant)
        ].sort_values("Produktionsjahr")
        if sub.empty:
            continue
        trend_figure.add_trace(
            go.Scatter(
                x=sub["Produktionsjahr"],
                y=sub["Quote"],
                mode="lines+markers",
                name=f"{plant} · {role}",
                line=dict(
                    color=ROLE_COLORS[role],
                    width=2,
                    dash=PLANT_LINESTYLES.get(plant, "solid"),
                ),
                marker=dict(size=7, color=ROLE_COLORS[role]),
                customdata=np.stack([sub["Anzahl"], sub["Fehler"]], axis=1),
                hovertemplate=(
                    "<b>%{fullData.name}</b><br>Year: %{x:.0f}"
                    "<br>Rate: %{y:.1%}"
                    "<br>Defective: %{customdata[1]:,.0f}"
                    "<br>Items: %{customdata[0]:,.0f}<extra></extra>"
                ),
                visible=False,
            )
        )
        trace_views.append(role)

view_titles = {
    "Overall": "Effective vehicle defect rate by production year",
    "Engine": "Engine defect rate by production year",
    "Seats": "Seat defect rate by production year",
    "Transmission": "Transmission defect rate by production year",
    "Body": "Body defect rate by production year",
    "All": "Vehicle and component-role defect rates by production year",
}


def values_for_view(view):
    if view == "Overall":
        return yearly_quality["Effektive_Quote"]
    if view == "All":
        return pd.concat(
            [yearly_quality["Effektive_Quote"], yearly_role["Quote"]], ignore_index=True
        )
    return yearly_role.loc[yearly_role["Role"] == view, "Quote"]


buttons = []
for view in ["Overall", "Engine", "Seats", "Transmission", "Body", "All"]:
    if view == "All":
        visible = [True] * len(trace_views)
    else:
        visible = [tag == view for tag in trace_views]
    yaxis = {"title": {"text": "Defect rate"}}
    yaxis.update(rate_axis(values_for_view(view)))
    buttons.append(
        dict(
            label=view,
            method="update",
            args=[
                {"visible": visible},
                {"title": {"text": view_titles[view]}, "yaxis": yaxis},
            ],
        )
    )

overall_yaxis = {"title": {"text": "Defect rate"}}
overall_yaxis.update(rate_axis(values_for_view("Overall")))

trend_figure.update_layout(
    title=view_titles["Overall"],
    yaxis=overall_yaxis,
    updatemenus=[
        dict(
            type="dropdown",
            buttons=buttons,
            x=1,
            xanchor="right",
            y=1.16,
            yanchor="top",
            showactive=True,
        )
    ],
)
trend_figure.update_xaxes(title="Production year", dtick=1)
style_figure(trend_figure, height=500)
trend_figure.update_layout(margin=dict(l=50, r=30, t=100, b=50))
trend_figure


# 8. Defect Sources at Supplier Level

The OEM ranking does not say which component is weak. The compact tables from Section 5.14 (`component_plant_quality`, `part_plant_quality`) give own defect rates at Tier-1 and Tier-2 plants. They are not a vehicle-level bill of materials.


In [ ]:
component_quality = load_or_reuse(
    "component_plant_quality", "component_plant_quality.csv"
).copy()
part_quality = load_or_reuse("part_plant_quality", "part_plant_quality.csv").copy()

component_quality["Werk_Label"] = [
    plant_label(werk, city)
    for werk, city in zip(component_quality["Werk"], component_quality["ORT"])
]
part_quality["Werk_Label"] = [
    plant_label(werk, city)
    for werk, city in zip(part_quality["Werk"], part_quality["ORT"])
]
component_quality["Quote_pct"] = component_quality["Fehlerquote"] * 100
part_quality["Quote_pct"] = part_quality["Fehlerquote"] * 100

print(
    f"{len(component_quality)} component plant-type rows, "
    f"{len(part_quality)} part plant-type rows."
)

part_by_type = (
    part_quality.groupby("Einzelteiltyp", dropna=False)
    .agg(Anzahl=("Anzahl", "sum"), Fehler=("Fehler", "sum"))
    .reset_index()
)
part_by_type["Fehlerquote"] = part_by_type["Fehler"] / part_by_type["Anzahl"]
part_by_type["Quote_pct"] = part_by_type["Fehlerquote"] * 100
part_by_type = part_by_type.sort_values("Fehlerquote", ascending=False).reset_index(
    drop=True
)


## 8.1 Component Types

**What the chart shows.** Own defect rate of each component type, pooled over producing plants, coloured by role (Engine, Seats, Transmission, Body). K4 is the Type 11 body, K6 the Type 21 body.

**Why it matters.** This is the first cut of the supplier layer: which types fail often, before looking at individual plants. Engine and some seat/transmission variants sit well above the ~10% body rate.


In [ ]:
component_by_type = (
    component_quality.groupby(["Komponententyp", "Komponentenrolle"], dropna=False)
    .agg(Anzahl=("Anzahl", "sum"), Fehler=("Fehler", "sum"))
    .reset_index()
)
component_by_type["Fehlerquote"] = (
    component_by_type["Fehler"] / component_by_type["Anzahl"]
)
component_by_type["Quote_pct"] = component_by_type["Fehlerquote"] * 100
component_by_type["Role"] = component_by_type["Komponentenrolle"].map(ROLE_LABELS)
component_by_type = component_by_type.sort_values(
    "Fehlerquote", ascending=False
).reset_index(drop=True)

component_type_figure = px.bar(
    component_by_type,
    x="Komponententyp",
    y="Quote_pct",
    color="Role",
    color_discrete_map=ROLE_COLORS,
    category_orders={"Role": ROLE_ORDER},
    custom_data=["Anzahl", "Fehler"],
    title="Component defect rate by type",
    labels={
        "Komponententyp": "Component type",
        "Quote_pct": "Defect rate [%]",
        "Role": "Role",
    },
)
component_type_figure.update_traces(
    hovertemplate=(
        "<b>%{x}</b><br>Rate: %{y:.2f} %"
        "<br>Items: %{customdata[0]:,.0f}"
        "<br>Defects: %{customdata[1]:,.0f}<extra></extra>"
    )
)
style_figure(component_type_figure)
component_type_figure.update_yaxes(
    title="Defect rate [%]",
    **rate_axis(component_by_type["Quote_pct"], percent_points=True),
)
component_type_figure.show()
component_by_type


## 8.2 Component Rates by Plant and Type

**What the chart shows.** The same rates, now split by producing plant. Some plants make more than one type; some types are made in more than one plant.

**Why it matters.** This is where the audit scope becomes concrete. K2LE1 and K2LE2 are much worse in Dortmund than in Ingolstadt. K3AG1 is much worse in Schweinfurt than in Hannover or Trunkelsberg. Those plant–type pairs are the ones to follow up, not a plant average.


In [ ]:
component_plant_figure = px.bar(
    component_quality.sort_values(["Komponententyp", "Werk_Label"]),
    x="Komponententyp",
    y="Quote_pct",
    color="Werk_Label",
    barmode="group",
    custom_data=["Anzahl", "Fehler"],
    title="Component defect rate by type and producing plant",
    labels={
        "Komponententyp": "Component type",
        "Quote_pct": "Defect rate [%]",
        "Werk_Label": "Tier-1 plant",
    },
)
component_plant_figure.update_traces(
    hovertemplate=(
        "<b>%{fullData.name}</b><br>%{x}: %{y:.2f} %"
        "<br>Items: %{customdata[0]:,.0f}"
        "<br>Defects: %{customdata[1]:,.0f}<extra></extra>"
    )
)
style_figure(component_plant_figure, height=500)
component_plant_figure.update_yaxes(
    title="Defect rate [%]",
    **rate_axis(component_quality["Quote_pct"], percent_points=True),
)
component_plant_figure.show()

component_quality.sort_values("Fehlerquote", ascending=False)[
    ["Werk_Label", "Komponententyp", "Komponentenrolle", "Anzahl", "Fehler", "Fehlerquote"]
].reset_index(drop=True)


## 8.3 Supply Chain

**What the chart shows.** A type-level chain part → component → vehicle type, using the mapping from Section 5.1. Left are part types, in the middle component types, on the right the two vehicle types. Node colour is the component role; part nodes use a lighter shade of the same colour. Larger nodes mean a higher own defect rate. A gold outline marks the plant-to-plant outliers from Section 8.2. Hover lists the producing plants.

The layout was generated with AI support as a readable overview of that mapping. The rates and plant lists still come from the tables above. It is not a shipment-level trace of individual part IDs.

**Why it matters.** It puts T01, the weak component types, and Type 21 on one page. That is only orientation for the audit visit; the ranking of OEM plants still comes from Section 7.


In [ ]:
# Fallback if section 4.1 was not run in this kernel.
if "component_part_types" not in globals():
    component_part_types = {
        "K4": ["T30", "T31", "T32"],
        "K3AG1": ["T21", "T24", "T25"],
        "K3SG1": ["T21", "T22", "T23"],
        "K2LE1": ["T11", "T14", "T15"],
        "K2ST1": ["T11", "T12", "T13"],
        "K1BE1": ["T01", "T02", "T03", "T04"],
        "K1DI1": ["T01", "T02", "T05", "T06"],
        "K6": ["T34", "T35", "T36", "T37"],
        "K3AG2": ["T21", "T24", "T27"],
        "K3SG2": ["T21", "T22", "T26"],
        "K2LE2": ["T16", "T19", "T20"],
        "K2ST2": ["T16", "T17", "T18"],
        "K1BE2": ["T01", "T02", "T07", "T08"],
        "K1DI2": ["T01", "T02", "T09", "T10"],
    }


def vehicle_type_of(komponententyp):
    if komponententyp == "K4":
        return "Type 11"
    if komponententyp == "K6":
        return "Type 21"
    if str(komponententyp).endswith("1"):
        return "Type 11"
    return "Type 21"


def plants_hover(table, type_column, type_name):
    rows = table.loc[table[type_column] == type_name].sort_values(
        "Fehlerquote", ascending=False
    )
    if rows.empty:
        return "no plant data"
    return "<br>".join(
        f"{werk}: {quote:.2f} %"
        for werk, quote in zip(rows["Werk_Label"], rows["Quote_pct"])
    )


component_role_map = (
    component_by_type.drop_duplicates("Komponententyp")
    .set_index("Komponententyp")["Role"]
    .to_dict()
)

part_role_map = {}
for komponententyp, part_types in component_part_types.items():
    role = component_role_map.get(komponententyp)
    for part_type in part_types:
        part_role_map.setdefault(part_type, role)

# Component types where plants disagree by at least 5 percentage points.
outlier_notes = {}
for komponententyp, grp in component_quality.groupby("Komponententyp"):
    if len(grp) < 2:
        continue
    top = grp.loc[grp["Fehlerquote"].idxmax()]
    low = grp.loc[grp["Fehlerquote"].idxmin()]
    gap = top["Fehlerquote"] - low["Fehlerquote"]
    if gap >= 0.05:
        outlier_notes[komponententyp] = (
            f"Plant outlier: {top['Werk_Label']} {top['Fehlerquote'] * 100:.1f} % "
            f"vs {low['Werk_Label']} {low['Fehlerquote'] * 100:.1f} %"
        )

part_types_ordered = sorted(
    part_by_type["Einzelteiltyp"], key=lambda name: int(str(name).lstrip("T"))
)
component_types_ordered = sorted(component_by_type["Komponententyp"].astype(str))
vehicle_types_ordered = ["Type 11", "Type 21"]

node_labels = part_types_ordered + component_types_ordered + vehicle_types_ordered
node_index = {name: i for i, name in enumerate(node_labels)}

part_rate_map = part_by_type.set_index("Einzelteiltyp")["Fehlerquote"]
component_rate_map = component_by_type.set_index("Komponententyp")["Fehlerquote"]
vehicle_rate_map = (
    analysis_vehicles.groupby("Fahrzeugtyp")["Effektiver_Fehler"].mean()
)

HIGH_RATE = 0.15

node_colors = []
node_line_colors = []
node_line_widths = []
node_hover = []
display_labels = []

for name in node_labels:
    extra = ""
    outline = "rgba(0,0,0,0.15)"
    outline_w = 0.5
    role = None

    if name in part_rate_map.index:
        rate = float(part_rate_map.loc[name])
        role = part_role_map.get(name)
        fill = mix_with_white(ROLE_COLORS.get(role, "#7F8C8D"), 0.55)
        if rate >= HIGH_RATE:
            outline, outline_w = "#1B1B1B", 3
            extra = "<br>High part rate"
        hover = (
            f"Part type {name}"
            + (f"<br>Role: {role}" if role else "")
            + f"<br>Rate: {rate * 100:.2f} %<br>"
            + plants_hover(part_quality, "Einzelteiltyp", name)
            + extra
        )
    elif name in component_rate_map.index:
        rate = float(component_rate_map.loc[name])
        role = component_role_map.get(name)
        fill = ROLE_COLORS.get(role, "#7F8C8D")
        if name in outlier_notes:
            outline, outline_w = "#D4A017", 4
            extra = "<br>" + outlier_notes[name]
        elif rate >= HIGH_RATE:
            outline, outline_w = "#1B1B1B", 3
            extra = "<br>High component rate"
        hover = (
            f"Component type {name}"
            + (f"<br>Role: {role}" if role else "")
            + f"<br>Rate: {rate * 100:.2f} %<br>"
            + plants_hover(component_quality, "Komponententyp", name)
            + extra
        )
    else:
        rate = float(vehicle_rate_map.loc[name])
        fill = "#5D6D7E"
        if rate >= float(vehicle_rate_map.max()) - 1e-12:
            outline, outline_w = "#1B1B1B", 3
            extra = "<br>Highest vehicle rate"
        hover = f"{name}<br>Effective vehicle rate: {rate * 100:.2f} %{extra}"

    mark = " *" if name in outlier_notes else ""
    display_labels.append(name + mark)
    node_colors.append(fill)
    node_line_colors.append(outline)
    node_line_widths.append(outline_w)
    node_hover.append(hover)

link_source = []
link_target = []
link_value = []
link_base_colors = []
link_hot_colors = []

part_targets = {}
for komponententyp, part_types in component_part_types.items():
    if komponententyp not in node_index:
        continue
    for part_type in part_types:
        if part_type in node_index:
            part_targets.setdefault(part_type, 0)
            part_targets[part_type] += 1

for komponententyp, part_types in component_part_types.items():
    if komponententyp not in node_index:
        continue
    role = component_role_map.get(komponententyp)
    role_color = ROLE_COLORS.get(role, "#7F8C8D")
    comp_rate = float(component_rate_map.loc[komponententyp])
    for part_type in part_types:
        if part_type not in node_index:
            continue
        part_rate = float(part_rate_map.loc[part_type])
        n_out = max(part_targets.get(part_type, 1), 1)
        link_source.append(node_index[part_type])
        link_target.append(node_index[komponententyp])
        # Box size in a Sankey comes from the flow. Split a part's rate
        # across its component links so T01 is large because of its rate,
        # not because it feeds four engines.
        link_value.append(max(part_rate, 0.02) / n_out)
        link_base_colors.append(hex_to_rgba(role_color, 0.55))
        link_hot_colors.append(role_color)
    vehicle_type = vehicle_type_of(komponententyp)
    link_source.append(node_index[komponententyp])
    link_target.append(node_index[vehicle_type])
    link_value.append(max(comp_rate, 0.02))
    link_base_colors.append(hex_to_rgba(role_color, 0.55))
    link_hot_colors.append(role_color)

n_nodes = len(node_labels)
outgoing = [[] for _ in range(n_nodes)]
incoming = [[] for _ in range(n_nodes)]
for i, (src, tgt) in enumerate(zip(link_source, link_target)):
    outgoing[src].append((i, tgt))
    incoming[tgt].append((i, src))

path_links_by_node = []
for node in range(n_nodes):
    links = set()
    stack = [node]
    seen = set()
    while stack:
        current = stack.pop()
        if current in seen:
            continue
        seen.add(current)
        for link_i, tgt in outgoing[current]:
            links.add(link_i)
            stack.append(tgt)
    stack = [node]
    seen = set()
    while stack:
        current = stack.pop()
        if current in seen:
            continue
        seen.add(current)
        for link_i, src in incoming[current]:
            links.add(link_i)
            stack.append(src)
    path_links_by_node.append(sorted(links))

supply_chain_figure = go.Figure(
    go.Sankey(
        arrangement="snap",
        node=dict(
            label=display_labels,
            color=node_colors,
            line=dict(color=node_line_colors, width=node_line_widths),
            customdata=node_hover,
            hovertemplate="%{customdata}<extra></extra>",
            pad=16,
            thickness=22,
        ),
        link=dict(
            source=link_source,
            target=link_target,
            value=link_value,
            color=link_base_colors,
        ),
    )
)
supply_chain_figure.update_layout(
    title="Type-level supply chain: part → component → vehicle type",
    font=dict(size=12),
    height=820,
    margin=dict(l=20, r=20, t=70, b=20),
    paper_bgcolor="white",
    hovermode="closest",
)
supply_chain_figure


# 9. Audit Recommendation

The next process audit should take place at **O21 Göttingen**. It has the highest effective field defect rate. Nürnberg affects more customers in absolute terms, but the task asks for a process audit, so the relative rate comes first. Within Type 11, O11 and O12 are essentially tied.

The visit should not stop at vehicle final assembly. Installed engines sit above body and transmission in the time series. At supplier level three follow-ups stand out:

- seats **K2LE1 / K2LE2** in Dortmund (vs Ingolstadt)
- transmission **K3AG1** in Schweinfurt (vs Hannover / Trunkelsberg)
- part **T01**, especially Fürth

O21 produces only Type 21. The data support auditing O21 first; they do not prove that assembly in Göttingen is worse than assembly in Nürnberg or Bonn. The charts say where to look. They do not replace a technical root-cause analysis on site.
